In [ ]:
# Repository bootstrap — portable local/Jupyter execution
from pathlib import Path
import os, sys

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "koa_severity").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from koa_severity.data import get_dataset_path
DATASET_PATH = get_dataset_path()
OUTPUTS_DIR = PROJECT_ROOT / "results"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
OUTPUTS_DIR.mkdir(exist_ok=True)
CHECKPOINTS_DIR.mkdir(exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU available")

In [ ]:

# Download latest version
path = get_dataset_path()

print("Path to dataset files:", path)

In [ ]:
import os
from collections import Counter

print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)

print("Dataset path:")
print(path)

# --------------------------------------------------
# 1. List top-level items
# --------------------------------------------------

print("\nTop-level contents:")

for item in os.listdir(path):
    full_path = os.path.join(path, item)

    if os.path.isdir(full_path):
        print(f"📁 {item}/")
    else:
        print(f"📄 {item}")


# --------------------------------------------------
# 2. Count files by extension
# --------------------------------------------------

extensions = Counter()

for root, dirs, files in os.walk(path):
    for file in files:
        ext = os.path.splitext(file)[1].lower()

        if ext:
            extensions[ext] += 1

print("\nFile types:")

for ext, count in extensions.items():
    print(f"{ext}: {count}")


# --------------------------------------------------
# 3. Show folder structure
# --------------------------------------------------

print("\nFolder structure:")

for root, dirs, files in os.walk(path):

    level = root.replace(path, "").count(os.sep)

    if level > 2:
        continue

    indent = "    " * level

    print(f"{indent}📁 {os.path.basename(root)}/")

    for file in files[:5]:
        print(f"{indent}    📄 {file}")

    if len(files) > 5:
        print(f"{indent}    ... ({len(files)} files)")


# --------------------------------------------------
# 4. Count images
# --------------------------------------------------

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_count = 0

for root, dirs, files in os.walk(path):
    for file in files:
        if os.path.splitext(file)[1].lower() in image_extensions:
            image_count += 1

print("\nTotal image files:", image_count)

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

DATASET_PATH = path

classes = ["0", "1", "2", "3", "4"]

fig, axes = plt.subplots(5, 5, figsize=(15, 15))

for row, cls in enumerate(classes):

    class_dir = os.path.join(DATASET_PATH, "train", cls)

    image_files = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith(".png")
    ]

    samples = random.sample(image_files, 5)

    for col, filename in enumerate(samples):

        image_path = os.path.join(class_dir, filename)
        image = Image.open(image_path)

        axes[row, col].imshow(image, cmap="gray")
        axes[row, col].axis("off")

        if col == 0:
            axes[row, col].set_ylabel(
                f"KL {cls}",
                fontsize=14
            )

        axes[row, col].set_title(filename, fontsize=8)

plt.tight_layout()

output_path = "./knee_samples.png"
plt.savefig(output_path, dpi=200, bbox_inches="tight")

plt.show()

print("Saved to:", output_path)

In [ ]:
import os
import re
from collections import defaultdict

DATASET_PATH = path

splits = ["train", "val", "test", "auto_test"]

# --------------------------------------------------
# 1. Extract ID from filename
# --------------------------------------------------

def extract_id(filename):
    """
    Examples:
        9268509L.png   -> 9268509
        9563713R.png   -> 9563713
        9531901_2.png  -> 9531901
        9768219_1.png  -> 9768219
    """

    name = os.path.splitext(filename)[0]

    match = re.match(r"(\d+)", name)

    if match:
        return match.group(1)

    return None


# --------------------------------------------------
# 2. Collect IDs for each split
# --------------------------------------------------

split_ids = defaultdict(set)
split_files = defaultdict(list)

for split in splits:

    split_path = os.path.join(DATASET_PATH, split)

    for root, dirs, files in os.walk(split_path):

        for file in files:

            if not file.lower().endswith(".png"):
                continue

            patient_id = extract_id(file)

            if patient_id is not None:
                split_ids[split].add(patient_id)
                split_files[split].append((patient_id, file))


# --------------------------------------------------
# 3. Basic statistics
# --------------------------------------------------

print("=" * 70)
print("UNIQUE ID ANALYSIS")
print("=" * 70)

for split in splits:

    print(f"{split:12s} -> "
          f"{len(split_files[split]):5d} images | "
          f"{len(split_ids[split]):5d} unique IDs")


# --------------------------------------------------
# 4. Find overlap between splits
# --------------------------------------------------

print("\n" + "=" * 70)
print("ID OVERLAP BETWEEN SPLITS")
print("=" * 70)

for i in range(len(splits)):

    for j in range(i + 1, len(splits)):

        split_a = splits[i]
        split_b = splits[j]

        overlap = split_ids[split_a] & split_ids[split_b]

        print(
            f"{split_a:12s} vs {split_b:12s} -> "
            f"{len(overlap)} shared IDs"
        )


# --------------------------------------------------
# 5. Show examples of leakage
# --------------------------------------------------

print("\n" + "=" * 70)
print("EXAMPLES OF SHARED IDs")
print("=" * 70)

for i in range(len(splits)):

    for j in range(i + 1, len(splits)):

        split_a = splits[i]
        split_b = splits[j]

        overlap = sorted(
            split_ids[split_a] & split_ids[split_b]
        )

        if overlap:

            print(f"\n{split_a} <-> {split_b}")
            print("Number of shared IDs:", len(overlap))
            print("Examples:", overlap[:20])

In [ ]:
from collections import Counter

print("=" * 70)
print("IMAGES PER ID")
print("=" * 70)

all_id_counts = Counter()

for split in splits:
    for patient_id, filename in split_files[split]:
        all_id_counts[patient_id] += 1

print("Total unique IDs:", len(all_id_counts))

print("\nDistribution of number of images per ID:")

distribution = Counter(all_id_counts.values())

for n_images, count in sorted(distribution.items()):
    print(f"{n_images} image(s) -> {count} IDs")


print("\nExamples of IDs with multiple images:")
print("-" * 70)

shown = 0

for patient_id, count in all_id_counts.items():

    if count > 1:

        examples = []

        for split in splits:
            for pid, filename in split_files[split]:

                if pid == patient_id:
                    examples.append(
                        f"{split}/{filename}"
                    )

        print(patient_id, "->", examples)

        shown += 1

        if shown >= 20:
            break

In [ ]:
import os
from collections import Counter, defaultdict

print("=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

for split in ["train", "val", "test", "auto_test"]:

    split_path = os.path.join(DATASET_PATH, split)

    counts = Counter()

    for class_name in ["0", "1", "2", "3", "4"]:

        class_path = os.path.join(split_path, class_name)

        count = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith(".png")
        ])

        counts[class_name] = count

    total = sum(counts.values())

    print(f"\n{split}")
    print("-" * 40)

    for cls, count in counts.items():

        percentage = 100 * count / total

        print(
            f"KL {cls}: "
            f"{count:4d} images "
            f"({percentage:5.2f}%)"
        )

    print(f"Total: {total}")


# --------------------------------------------------
# Check whether L/R of the same ID have
# the same or different labels
# --------------------------------------------------

print("\n" + "=" * 70)
print("LEFT / RIGHT LABEL CONSISTENCY")
print("=" * 70)

for split in ["train", "val", "test"]:

    split_path = os.path.join(DATASET_PATH, split)

    patient_labels = defaultdict(dict)

    for cls in ["0", "1", "2", "3", "4"]:

        class_path = os.path.join(split_path, cls)

        for filename in os.listdir(class_path):

            if not filename.endswith(".png"):
                continue

            patient_id = extract_id(filename)

            name = os.path.splitext(filename)[0]

            if name.endswith("L"):
                side = "L"
            elif name.endswith("R"):
                side = "R"
            else:
                side = "?"

            patient_labels[patient_id][side] = cls

    same_label = 0
    different_label = 0

    examples = []

    for patient_id, labels in patient_labels.items():

        if "L" in labels and "R" in labels:

            if labels["L"] == labels["R"]:
                same_label += 1
            else:
                different_label += 1

                if len(examples) < 10:
                    examples.append(
                        (
                            patient_id,
                            labels["L"],
                            labels["R"]
                        )
                    )

    print(f"\n{split}")
    print(f"Same L/R label:      {same_label}")
    print(f"Different L/R label: {different_label}")

    if examples:

        print("Examples of different grades:")

        for patient_id, left, right in examples:

            print(
                f"  {patient_id}: "
                f"L=KL{left}, R=KL{right}"
            )

In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# مسیرهای داده
TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR   = os.path.join(DATASET_PATH, "val")
TEST_DIR  = os.path.join(DATASET_PATH, "test")

# فعلاً فقط preprocessing پایه
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

# ساخت Dataset
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=transform
)

# نمایش mapping کلاس‌ها
print("Class mapping:")
print(train_dataset.class_to_idx)

print("\nNumber of images:")
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label:", label)
print("Pixel minimum:", image.min().item())
print("Pixel maximum:", image.max().item())

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# گرفتن اولین batch
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels)

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: Not available")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: Not available")

In [ ]:
import matplotlib.pyplot as plt

# گرفتن یک batch
images, labels = next(iter(train_loader))

class_names = {
    0: "Normal",
    1: "Doubtful",
    2: "Mild",
    3: "Moderate",
    4: "Severe"
}

plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)

    # [C,H,W] → [H,W,C]
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(class_names[labels[i].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np

# تعداد نمونه‌های هر کلاس در Train
class_counts = np.bincount(train_dataset.targets)

print("Class counts:")
for i, count in enumerate(class_counts):
    print(f"Class {i}: {count}")

# محاسبه وزن کلاس‌ها
class_weights = len(train_dataset) / (len(class_counts) * class_counts)

print("\nClass weights:")
for i, weight in enumerate(class_weights):
    print(f"Class {i}: {weight:.4f}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# --------------------------------------------------
# 1. انتخاب device
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# --------------------------------------------------
# 2. بارگذاری ResNet18 از قبل آموزش‌دیده
# --------------------------------------------------

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)


# --------------------------------------------------
# 3. تغییر لایه آخر برای 5 کلاس
# --------------------------------------------------

num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    5
)


# --------------------------------------------------
# 4. انتقال مدل به GPU
# --------------------------------------------------

model = model.to(device)


# --------------------------------------------------
# 5. ساخت Class-Weighted Loss
# --------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


print("\nModel ready.")
print("Number of output classes:", model.fc.out_features)
print("Loss:", criterion)

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("Optimizer:", optimizer)
print("Learning rate:", 1e-4)

In [ ]:
NUM_EPOCHS = 5

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


for epoch in range(NUM_EPOCHS):

    # ==================================================
    # TRAIN
    # ==================================================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # انتقال داده به GPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # صفر کردن gradientها
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # محاسبه Loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # به‌روزرسانی وزن‌های مدل
        optimizer.step()

        # جمع کردن Loss
        running_loss += loss.item() * images.size(0)

        # محاسبه accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total


    # ==================================================
    # VALIDATION
    # ==================================================
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = running_loss / total
    epoch_val_accuracy = correct / total


    # ذخیره نتایج
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    train_accuracies.append(epoch_train_accuracy)
    val_accuracies.append(epoch_val_accuracy)


    # نمایش نتیجه Epoch
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score
)

# مدل در حالت evaluation
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)

        _, predictions = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())


all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)


# --------------------------------------------
# Class names
# --------------------------------------------

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]


# --------------------------------------------
# Classification report
# --------------------------------------------

print("Classification Report:\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# --------------------------------------------
# Balanced Accuracy
# --------------------------------------------

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

print(f"Balanced Accuracy: {balanced_acc:.4f}")


# --------------------------------------------
# Macro F1
# --------------------------------------------

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

print(f"Macro F1: {macro_f1:.4f}")


# --------------------------------------------
# Confusion Matrix
# --------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)


# --------------------------------------------
# Plot Confusion Matrix
# --------------------------------------------

plt.figure(figsize=(7, 6))

plt.imshow(cm)

plt.title("Validation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.xticks(
    range(5),
    class_names,
    rotation=30
)

plt.yticks(
    range(5),
    class_names
)

# نمایش عدد داخل هر خانه
for i in range(5):
    for j in range(5):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import f1_score

# --------------------------------------------------
# 1. Create a fresh ResNet18
# --------------------------------------------------

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 5)

model = model.to(device)

# --------------------------------------------------
# 2. Loss with class weights
# --------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

# --------------------------------------------------
# 3. Optimizer
# --------------------------------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

# --------------------------------------------------
# 4. Training settings
# --------------------------------------------------

NUM_EPOCHS = 10

best_macro_f1 = 0.0
best_epoch = 0

BEST_MODEL_PATH = "./best_resnet18_oa.pth"

print("Starting training...")
print("-" * 70)

# --------------------------------------------------
# 5. Training loop
# --------------------------------------------------

for epoch in range(NUM_EPOCHS):

    # ==========================
    # TRAIN
    # ==========================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, predictions = torch.max(outputs, 1)

        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ==========================
    # VALIDATION
    # ==========================

    model.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predictions = torch.max(outputs, 1)

            val_total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(all_predictions, all_labels)
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    # ==========================
    # SAVE BEST MODEL
    # ==========================

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1
        best_epoch = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_MODEL_PATH
        )

        saved = " <-- BEST MODEL SAVED"

    else:
        saved = ""

    # ==========================
    # PRINT RESULTS
    # ==========================

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)
print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Macro F1: {best_macro_f1:.4f}")
print(f"Saved model: {BEST_MODEL_PATH}")

In [ ]:
from torchvision import transforms

transform_normalized = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset_norm = datasets.ImageFolder(
    TRAIN_DIR,
    transform=transform_normalized
)

val_dataset_norm = datasets.ImageFolder(
    VAL_DIR,
    transform=transform_normalized
)

test_dataset_norm = datasets.ImageFolder(
    TEST_DIR,
    transform=transform_normalized
)

print("Train:", len(train_dataset_norm))
print("Val:", len(val_dataset_norm))
print("Test:", len(test_dataset_norm))

In [ ]:
train_loader_norm = DataLoader(
    train_dataset_norm,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_norm = DataLoader(
    val_dataset_norm,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_norm = DataLoader(
    test_dataset_norm,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import f1_score

# --------------------------------------------------
# 1. Create a fresh ResNet18
# --------------------------------------------------

weights = models.ResNet18_Weights.DEFAULT

model_norm = models.resnet18(weights=weights)

num_features = model_norm.fc.in_features
model_norm.fc = nn.Linear(num_features, 5)

model_norm = model_norm.to(device)

# --------------------------------------------------
# 2. Class-weighted loss
# --------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion_norm = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

# --------------------------------------------------
# 3. Optimizer
# --------------------------------------------------

optimizer_norm = optim.Adam(
    model_norm.parameters(),
    lr=1e-4
)

# --------------------------------------------------
# 4. Training settings
# --------------------------------------------------

NUM_EPOCHS = 10

best_macro_f1_norm = 0.0
best_epoch_norm = 0

BEST_MODEL_PATH_NORM = "./best_resnet18_oa_normalized.pth"

print("Starting Experiment B...")
print("-" * 70)

# --------------------------------------------------
# 5. Training loop
# --------------------------------------------------

for epoch in range(NUM_EPOCHS):

    # ==========================
    # TRAIN
    # ==========================

    model_norm.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_norm:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_norm.zero_grad()

        outputs = model_norm(images)

        loss = criterion_norm(outputs, labels)

        loss.backward()

        optimizer_norm.step()

        train_loss += loss.item() * images.size(0)

        _, predictions = torch.max(outputs, 1)

        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ==========================
    # VALIDATION
    # ==========================

    model_norm.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_norm:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model_norm(images)

            loss = criterion_norm(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predictions = torch.max(outputs, 1)

            val_total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(all_predictions, all_labels)
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    # ==========================
    # SAVE BEST MODEL
    # ==========================

    if macro_f1 > best_macro_f1_norm:

        best_macro_f1_norm = macro_f1
        best_epoch_norm = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model_norm.state_dict(),
                "optimizer_state_dict": optimizer_norm.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_MODEL_PATH_NORM
        )

        saved = " <-- BEST MODEL SAVED"

    else:
        saved = ""

    # ==========================
    # PRINT RESULTS
    # ==========================

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)
print(f"Best Epoch: {best_epoch_norm}")
print(f"Best Validation Macro F1: {best_macro_f1_norm:.4f}")
print(f"Saved model: {BEST_MODEL_PATH_NORM}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score
)

# --------------------------------------------------
# Load BEST Experiment B model
# --------------------------------------------------

checkpoint = torch.load(
    "./best_resnet18_oa_normalized.pth",
    map_location=device,
    weights_only=False
)

model_norm.load_state_dict(
    checkpoint["model_state_dict"]
)

model_norm.eval()

print("Loaded best model from epoch:", checkpoint["epoch"])
print("Saved Macro F1:", checkpoint["macro_f1"])
print("Saved Val Accuracy:", checkpoint["val_acc"])

# --------------------------------------------------
# Evaluation
# --------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader_norm:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model_norm(images)

        _, predictions = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

# --------------------------------------------------
# Classification Report
# --------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# --------------------------------------------------
# Metrics
# --------------------------------------------------

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

# --------------------------------------------------
# Confusion Matrix
# --------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)

# --------------------------------------------------
# Plot
# --------------------------------------------------

plt.figure(figsize=(7, 6))

plt.imshow(cm)

plt.title("Experiment B - Validation Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.xticks(
    range(5),
    class_names,
    rotation=30
)

plt.yticks(
    range(5),
    class_names
)

for i in range(5):
    for j in range(5):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
from torchvision import transforms

train_transform_aug = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.Grayscale(num_output_channels=3),

    # Augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    # ImageNet normalization
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation/Test MUST NOT have augmentation
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms created successfully.")

In [ ]:
train_dataset_aug = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform_aug
)

val_dataset_aug = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform
)

test_dataset_aug = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)

print("Train:", len(train_dataset_aug))
print("Val:", len(val_dataset_aug))
print("Test:", len(test_dataset_aug))

In [ ]:
train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_aug = DataLoader(
    val_dataset_aug,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_aug = DataLoader(
    test_dataset_aug,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Augmented DataLoaders created successfully.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import f1_score

# --------------------------------------------------
# 1. Fresh ResNet18
# --------------------------------------------------

weights = models.ResNet18_Weights.DEFAULT

model_aug = models.resnet18(weights=weights)

num_features = model_aug.fc.in_features
model_aug.fc = nn.Linear(num_features, 5)

model_aug = model_aug.to(device)

# --------------------------------------------------
# 2. Class-weighted loss
# --------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion_aug = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

# --------------------------------------------------
# 3. Optimizer
# --------------------------------------------------

optimizer_aug = optim.Adam(
    model_aug.parameters(),
    lr=1e-4
)

# --------------------------------------------------
# 4. Training settings
# --------------------------------------------------

NUM_EPOCHS = 10

best_macro_f1_aug = 0.0
best_epoch_aug = 0

BEST_MODEL_PATH_AUG = "./best_resnet18_oa_augmented.pth"

print("Starting Experiment C...")
print("-" * 70)

# --------------------------------------------------
# 5. Training loop
# --------------------------------------------------

for epoch in range(NUM_EPOCHS):

    # ==========================
    # TRAIN
    # ==========================

    model_aug.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_aug:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_aug.zero_grad()

        outputs = model_aug(images)

        loss = criterion_aug(outputs, labels)

        loss.backward()

        optimizer_aug.step()

        train_loss += loss.item() * images.size(0)

        _, predictions = torch.max(outputs, 1)

        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ==========================
    # VALIDATION
    # ==========================

    model_aug.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_aug:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model_aug(images)

            loss = criterion_aug(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predictions = torch.max(outputs, 1)

            val_total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(all_predictions, all_labels)
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    # ==========================
    # SAVE BEST MODEL
    # ==========================

    if macro_f1 > best_macro_f1_aug:

        best_macro_f1_aug = macro_f1
        best_epoch_aug = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model_aug.state_dict(),
                "optimizer_state_dict": optimizer_aug.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_MODEL_PATH_AUG
        )

        saved = " <-- BEST MODEL SAVED"

    else:
        saved = ""

    # ==========================
    # PRINT RESULTS
    # ==========================

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)
print(f"Best Epoch: {best_epoch_aug}")
print(f"Best Validation Macro F1: {best_macro_f1_aug:.4f}")
print(f"Saved model: {BEST_MODEL_PATH_AUG}")

In [ ]:
import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score
)

# --------------------------------------------------
# 1. Load best Experiment C checkpoint
# --------------------------------------------------

checkpoint = torch.load(
    "./best_resnet18_oa_augmented.pth",
    map_location=device,
    weights_only=False
)

model_aug.load_state_dict(
    checkpoint["model_state_dict"]
)

model_aug.eval()

print("Best checkpoint loaded.")
print("Best epoch:", checkpoint["epoch"])
print("Saved Macro F1:", checkpoint["macro_f1"])
print("Saved Val Accuracy:", checkpoint["val_acc"])


# --------------------------------------------------
# 2. Evaluate on validation set
# --------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader_aug:

        images = images.to(device, non_blocking=True)

        outputs = model_aug(images)

        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(
            predictions.cpu().numpy()
        )


# --------------------------------------------------
# 3. Classification report
# --------------------------------------------------

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# --------------------------------------------------
# 4. Balanced Accuracy
# --------------------------------------------------

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")


# --------------------------------------------------
# 5. Confusion Matrix
# --------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
import cv2
import numpy as np
from PIL import Image
import torchvision.transforms as transforms


class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):

        # PIL → NumPy
        img = np.array(img)

        # اگر تصویر RGB بود، تبدیل به grayscale
        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # CLAHE
        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        img = clahe.apply(img)

        # NumPy → PIL
        return Image.fromarray(img)

In [ ]:
# ============================================================
# Experiment D — CLAHE + Normalization + Augmentation
# ============================================================

train_transform_clahe = transforms.Compose([

    # 1. Resize
    transforms.Resize((224, 224)),

    # 2. CLAHE
    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    # 3. Convert grayscale → 3 channels
    transforms.Grayscale(num_output_channels=3),

    # 4. Data augmentation
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    # 5. Tensor
    transforms.ToTensor(),

    # 6. ImageNet normalization
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Validation / Test
# ------------------------------------------------------------

eval_transform_clahe = transforms.Compose([

    transforms.Resize((224, 224)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(num_output_channels=3),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset_clahe = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform_clahe
)

val_dataset_clahe = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform_clahe
)

test_dataset_clahe = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform_clahe
)


# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader_clahe = DataLoader(
    train_dataset_clahe,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_clahe = DataLoader(
    val_dataset_clahe,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_clahe = DataLoader(
    test_dataset_clahe,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("Experiment D datasets and DataLoaders created successfully.")
print()
print("Train:", len(train_dataset_clahe))
print("Val:  ", len(val_dataset_clahe))
print("Test: ", len(test_dataset_clahe))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import f1_score


# ============================================================
# 1. Fresh ResNet18
# ============================================================

weights = models.ResNet18_Weights.DEFAULT

model_clahe = models.resnet18(weights=weights)

num_features = model_clahe.fc.in_features

model_clahe.fc = nn.Linear(
    num_features,
    5
)

model_clahe = model_clahe.to(device)


# ============================================================
# 2. Class-weighted loss
# ============================================================

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion_clahe = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ============================================================
# 3. Optimizer
# ============================================================

optimizer_clahe = optim.Adam(
    model_clahe.parameters(),
    lr=1e-4
)


# ============================================================
# 4. Training settings
# ============================================================

NUM_EPOCHS = 10

best_macro_f1_clahe = 0.0
best_epoch_clahe = 0

BEST_MODEL_PATH_CLAHE = (
    "./best_resnet18_oa_clahe.pth"
)


print("Starting Experiment D...")
print("-" * 70)


# ============================================================
# 5. Training loop
# ============================================================

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_clahe.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_clahe.zero_grad()

        outputs = model_clahe(images)

        loss = criterion_clahe(
            outputs,
            labels
        )

        loss.backward()

        optimizer_clahe.step()

        train_loss += (
            loss.item() * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss /= train_total

    train_acc = (
        train_correct / train_total
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_clahe.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_clahe(images)

            loss = criterion_clahe(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )


    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total


    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_clahe:

        best_macro_f1_clahe = macro_f1

        best_epoch_clahe = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    model_clahe.state_dict(),
                "optimizer_state_dict":
                    optimizer_clahe.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_MODEL_PATH_CLAHE
        )

        saved = " <-- BEST MODEL SAVED"

    else:

        saved = ""


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Epoch: {best_epoch_clahe}"
)

print(
    f"Best Validation Macro F1: "
    f"{best_macro_f1_clahe:.4f}"
)

print(
    f"Saved model: "
    f"{BEST_MODEL_PATH_CLAHE}"
)

In [ ]:
import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score
)

# ============================================================
# 1. Load best Experiment D checkpoint
# ============================================================

checkpoint = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_clahe.load_state_dict(
    checkpoint["model_state_dict"]
)

model_clahe.eval()

print("Best checkpoint loaded.")
print("Best epoch:", checkpoint["epoch"])
print("Saved Macro F1:", checkpoint["macro_f1"])
print("Saved Val Accuracy:", checkpoint["val_acc"])


# ============================================================
# 2. Validation evaluation
# ============================================================

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader_clahe:

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model_clahe(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )


# ============================================================
# 3. Classification report
# ============================================================

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 4. Main metrics
# ============================================================

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

print(
    f"Balanced Accuracy: {balanced_acc:.4f}"
)

print(
    f"Macro F1:          {macro_f1:.4f}"
)


# ============================================================
# 5. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models


# ============================================================
# Experiment E — Frozen Backbone
# ============================================================

weights = models.ResNet18_Weights.DEFAULT

model_frozen = models.resnet18(weights=weights)


# ------------------------------------------------------------
# Freeze entire backbone
# ------------------------------------------------------------

for param in model_frozen.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# Replace classifier
# ------------------------------------------------------------

num_features = model_frozen.fc.in_features

model_frozen.fc = nn.Linear(
    num_features,
    5
)


# Only the new FC layer should be trainable
model_frozen = model_frozen.to(device)


# ------------------------------------------------------------
# Class-weighted loss
# ------------------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion_frozen = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_frozen = optim.Adam(
    model_frozen.fc.parameters(),
    lr=1e-3
)


# ------------------------------------------------------------
# Check trainable parameters
# ------------------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in model_frozen.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model_frozen.parameters()
)

print("Experiment E model created.")
print()
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
from sklearn.metrics import f1_score

# ============================================================
# Experiment E — Frozen Backbone Training
# ============================================================

NUM_EPOCHS = 5

best_macro_f1_frozen = 0.0
best_epoch_frozen = 0

BEST_MODEL_PATH_FROZEN = (
    "./best_resnet18_oa_frozen.pth"
)

print("Starting Experiment E...")
print("-" * 70)


# ============================================================
# Training
# ============================================================

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_frozen.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_frozen.zero_grad()

        outputs = model_frozen(images)

        loss = criterion_frozen(
            outputs,
            labels
        )

        loss.backward()

        optimizer_frozen.step()

        train_loss += (
            loss.item() * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss /= train_total

    train_acc = (
        train_correct / train_total
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_frozen.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_frozen(images)

            loss = criterion_frozen(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )


    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total


    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_frozen:

        best_macro_f1_frozen = macro_f1
        best_epoch_frozen = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict":
                    model_frozen.state_dict(),
                "optimizer_state_dict":
                    optimizer_frozen.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_MODEL_PATH_FROZEN
        )

        saved = " <-- BEST MODEL SAVED"

    else:

        saved = ""


    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Epoch: {best_epoch_frozen}"
)

print(
    f"Best Validation Macro F1: "
    f"{best_macro_f1_frozen:.4f}"
)

print(
    f"Saved model: "
    f"{BEST_MODEL_PATH_FROZEN}"
)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models


# ============================================================
# Stage 1 — Start from best Experiment D
# ============================================================

weights = models.ResNet18_Weights.DEFAULT

model_ft = models.resnet18(weights=weights)

# Replace classifier
num_features = model_ft.fc.in_features

model_ft.fc = nn.Linear(
    num_features,
    5
)


# ============================================================
# Load best Experiment D checkpoint
# ============================================================

checkpoint_d = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_ft.load_state_dict(
    checkpoint_d["model_state_dict"]
)

model_ft = model_ft.to(device)


# ============================================================
# Freeze backbone
# ============================================================

for param in model_ft.parameters():
    param.requires_grad = False


# Unfreeze classifier
for param in model_ft.fc.parameters():
    param.requires_grad = True


# ============================================================
# Loss
# ============================================================

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion_ft = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ============================================================
# Optimizer
# ============================================================

optimizer_ft = optim.Adam(
    model_ft.fc.parameters(),
    lr=1e-4
)


# ============================================================
# Check
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model_ft.parameters()
    if p.requires_grad
)

print("Stage 1 model ready.")
print()
print("Loaded checkpoint epoch:",
      checkpoint_d["epoch"])

print("Loaded Macro F1:",
      checkpoint_d["macro_f1"])

print("Trainable parameters:",
      f"{trainable_params:,}")

In [ ]:
from sklearn.metrics import f1_score

# ============================================================
# Stage 1 — Classifier Warm-up
# ============================================================

NUM_EPOCHS_STAGE1 = 3

best_macro_f1_stage1 = checkpoint_d["macro_f1"]
best_epoch_stage1 = checkpoint_d["epoch"]

BEST_STAGE1_PATH = (
    "./best_resnet18_oa_stage1.pth"
)

# Save current best as starting point
torch.save(
    {
        "epoch": checkpoint_d["epoch"],
        "model_state_dict": model_ft.state_dict(),
        "optimizer_state_dict": optimizer_ft.state_dict(),
        "macro_f1": checkpoint_d["macro_f1"],
        "val_acc": checkpoint_d["val_acc"],
    },
    BEST_STAGE1_PATH
)

print("Starting Stage 1...")
print("-" * 70)


for epoch in range(NUM_EPOCHS_STAGE1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_ft.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_ft.zero_grad()

        outputs = model_ft(images)

        loss = criterion_ft(
            outputs,
            labels
        )

        loss.backward()

        optimizer_ft.step()

        train_loss += (
            loss.item() * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss /= train_total

    train_acc = (
        train_correct / train_total
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_ft.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_ft(images)

            loss = criterion_ft(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )


    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE IF IMPROVED
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_stage1:

        best_macro_f1_stage1 = macro_f1

        best_epoch_stage1 = (
            checkpoint_d["epoch"] + epoch + 1
        )

        torch.save(
            {
                "epoch": best_epoch_stage1,
                "model_state_dict":
                    model_ft.state_dict(),
                "optimizer_state_dict":
                    optimizer_ft.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_STAGE1_PATH
        )

        saved = " <-- NEW BEST"

    else:

        saved = ""


    print(
        f"Stage 1 Epoch [{epoch+1}/"
        f"{NUM_EPOCHS_STAGE1}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Macro F1 after Stage 1: "
    f"{best_macro_f1_stage1:.4f}"
)

In [ ]:
# ============================================================
# Stage 2 — Unfreeze Backbone
# ============================================================

# ------------------------------------------------------------
# 1. Load best Stage 1 checkpoint
# ------------------------------------------------------------

checkpoint_stage1 = torch.load(
    "./best_resnet18_oa_stage1.pth",
    map_location=device,
    weights_only=False
)

model_ft.load_state_dict(
    checkpoint_stage1["model_state_dict"]
)


# ------------------------------------------------------------
# 2. Unfreeze entire model
# ------------------------------------------------------------

for param in model_ft.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 3. Separate learning rates
# ------------------------------------------------------------

optimizer_stage2 = optim.Adam(
    [
        {
            "params": model_ft.layer1.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_ft.layer2.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_ft.layer3.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_ft.layer4.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_ft.fc.parameters(),
            "lr": 1e-5
        }
    ]
)


# ------------------------------------------------------------
# 4. Check trainable parameters
# ------------------------------------------------------------

trainable_params_stage2 = sum(
    p.numel()
    for p in model_ft.parameters()
    if p.requires_grad
)

print("Stage 2 ready.")
print()
print("Starting Macro F1:",
      checkpoint_stage1["macro_f1"])

print("Trainable parameters:",
      f"{trainable_params_stage2:,}")

print("Learning rate: 1e-5")

In [ ]:
from sklearn.metrics import f1_score

# ============================================================
# Stage 2 — Full Fine-Tuning
# ============================================================

NUM_EPOCHS_STAGE2 = 5

best_macro_f1_stage2 = checkpoint_stage1["macro_f1"]
best_epoch_stage2 = checkpoint_stage1["epoch"]

BEST_STAGE2_PATH = (
    "./best_resnet18_oa_finetuned.pth"
)

# ------------------------------------------------------------
# Save Stage 1 checkpoint as current best
# ------------------------------------------------------------

torch.save(
    {
        "epoch": checkpoint_stage1["epoch"],
        "model_state_dict":
            model_ft.state_dict(),
        "optimizer_state_dict":
            optimizer_stage2.state_dict(),
        "macro_f1":
            checkpoint_stage1["macro_f1"],
        "val_acc":
            checkpoint_stage1["val_acc"],
    },
    BEST_STAGE2_PATH
)

print("Starting Stage 2...")
print("-" * 70)


# ============================================================
# Training loop
# ============================================================

for epoch in range(NUM_EPOCHS_STAGE2):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_ft.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_stage2.zero_grad()

        outputs = model_ft(images)

        loss = criterion_ft(
            outputs,
            labels
        )

        loss.backward()

        optimizer_stage2.step()

        train_loss += (
            loss.item() * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss /= train_total

    train_acc = (
        train_correct / train_total
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_ft.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_ft(images)

            loss = criterion_ft(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )


    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total


    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_stage2:

        best_macro_f1_stage2 = macro_f1

        best_epoch_stage2 = (
            checkpoint_stage1["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "epoch": best_epoch_stage2,
                "model_state_dict":
                    model_ft.state_dict(),
                "optimizer_state_dict":
                    optimizer_stage2.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_STAGE2_PATH
        )

        saved = " <-- NEW BEST"

    else:

        saved = ""


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Stage 2 Epoch [{epoch+1}/"
        f"{NUM_EPOCHS_STAGE2}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Macro F1 after Stage 2: "
    f"{best_macro_f1_stage2:.4f}"
)

print(
    f"Saved model: {BEST_STAGE2_PATH}"
)

In [ ]:
# ============================================================
# Validation Evaluation — Best Stage 2 Model
# ============================================================

import torch
import numpy as np

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    mean_absolute_error,
    cohen_kappa_score
)

# ------------------------------------------------------------
# 1. Load best Stage 2 checkpoint
# ------------------------------------------------------------

BEST_STAGE2_PATH = "./best_resnet18_oa_finetuned.pth"

checkpoint = torch.load(
    BEST_STAGE2_PATH,
    map_location=device,
    weights_only=False
)

model_ft.load_state_dict(checkpoint["model_state_dict"])
model_ft = model_ft.to(device)
model_ft.eval()

print("=" * 70)
print("BEST STAGE 2 MODEL")
print("=" * 70)
print(f"Checkpoint epoch : {checkpoint['epoch']}")
print(f"Saved Macro F1   : {checkpoint['macro_f1']:.4f}")
print(f"Saved Val Acc    : {checkpoint['val_acc']:.4f}")
print("=" * 70)


# ------------------------------------------------------------
# 2. Predictions on validation set
# ------------------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader_clahe:

        images = images.to(device, non_blocking=True)

        outputs = model_ft(images)

        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())


# ------------------------------------------------------------
# 3. Basic metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

mae = mean_absolute_error(
    all_labels,
    all_predictions
)

qwk = cohen_kappa_score(
    all_labels,
    all_predictions,
    weights="quadratic"
)


# ------------------------------------------------------------
# 4. Print overall results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy          : {accuracy:.4f}")
print(f"Balanced Accuracy : {balanced_acc:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")
print(f"MAE               : {mae:.4f}")
print(f"QWK               : {qwk:.4f}")


# ------------------------------------------------------------
# 5. Per-class classification report
# ------------------------------------------------------------

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 6. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print("Rows = True class")
print("Cols = Predicted class\n")

print("             " + "  ".join(
    f"{name[:9]:>9}" for name in class_names
))

for i, row in enumerate(cm):

    print(
        f"{class_names[i]:<10}"
        + "  ".join(f"{value:9d}" for value in row)
    )

In [ ]:
# ============================================================
# Robustness Experiment — Seed 42
# ============================================================

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed set to: {SEED}")

In [ ]:
# ============================================================
# Robustness Experiment — Seed 42
# Build a completely fresh model
# ============================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

SEED = 42

# ------------------------------------------------------------
# Set seed
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# Fresh pretrained ResNet18
# ------------------------------------------------------------

weights = models.ResNet18_Weights.DEFAULT

model_seed42 = models.resnet18(weights=weights)

num_features = model_seed42.fc.in_features

model_seed42.fc = nn.Linear(
    num_features,
    5
)

model_seed42 = model_seed42.to(device)


# ------------------------------------------------------------
# Weighted loss
# ------------------------------------------------------------

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion_seed42 = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# Stage 1 optimizer
# Only classifier is trainable
# ------------------------------------------------------------

for param in model_seed42.parameters():
    param.requires_grad = False

for param in model_seed42.fc.parameters():
    param.requires_grad = True

optimizer_seed42_stage1 = optim.Adam(
    model_seed42.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check trainable parameters
# ------------------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in model_seed42.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model_seed42.parameters()
)

print("=" * 70)
print("SEED 42 MODEL READY")
print("=" * 70)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Stage 1 learning rate: 1e-4")
print(f"Seed                  : {SEED}")

print("=" * 70)

In [ ]:
# ============================================================
# Robustness Experiment — Seed 42
# Stage 1: Train classifier head only
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 3

best_macro_f1_seed42 = -1.0
best_epoch_seed42 = 0

BEST_SEED42_STAGE1 = (
    "./best_resnet18_oa_seed42_stage1.pth"
)

print("=" * 70)
print("SEED 42 — STAGE 1 TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model_seed42.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_seed42_stage1.zero_grad()

        outputs = model_seed42(images)

        loss = criterion_seed42(outputs, labels)

        loss.backward()

        optimizer_seed42_stage1.step()

        train_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total

    train_acc = (
        train_correct / train_total
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_seed42.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_seed42(images)

            loss = criterion_seed42(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_seed42:

        best_macro_f1_seed42 = macro_f1
        best_epoch_seed42 = epoch + 1

        torch.save(
            {
                "seed": SEED,
                "epoch": epoch + 1,
                "model_state_dict":
                    model_seed42.state_dict(),
                "optimizer_state_dict":
                    optimizer_seed42_stage1.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_SEED42_STAGE1
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Seed 42 Stage 1 Macro F1: "
    f"{best_macro_f1_seed42:.4f}"
)

print(
    f"Best epoch: {best_epoch_seed42}"
)

print(
    f"Saved model: {BEST_SEED42_STAGE1}"
)

In [ ]:
# ============================================================
# DEBUG — Check Seed 42 data pipeline
# ============================================================

print("=" * 70)
print("DATA PIPELINE CHECK")
print("=" * 70)

print("Train samples:", len(train_dataset_clahe))
print("Val samples  :", len(val_dataset_clahe))

print("\nClass mapping:")
print(train_dataset_clahe.class_to_idx)

print("\nFirst 20 training labels:")

labels_sample = []

for images, labels in train_loader_clahe:

    labels_sample.extend(
        labels.numpy().tolist()
    )

    if len(labels_sample) >= 20:
        break

print(labels_sample[:20])

print("\nLabel distribution in first batch:")

images, labels = next(iter(train_loader_clahe))

unique, counts = np.unique(
    labels.numpy(),
    return_counts=True
)

for cls, count in zip(unique, counts):
    print(
        f"Class {cls}: {count}"
    )

print("\nBatch shape:")
print(images.shape)

print("\nImage value range:")
print("Min:", images.min().item())
print("Max:", images.max().item())

print("=" * 70)

In [ ]:
# ============================================================
# Seed 42 — Rebuild model exactly like original experiment
# ============================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

SEED = 42

# Seed everything
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# IMPORTANT:
# Use normal cuDNN behavior, like the original experiments
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# ------------------------------------------------------------
# Fresh ResNet18
# ------------------------------------------------------------

model_seed42_test = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_seed42_test.fc = nn.Linear(
    model_seed42_test.fc.in_features,
    5
)

model_seed42_test = model_seed42_test.to(device)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_seed42_test.parameters():
    param.requires_grad = False

for param in model_seed42_test.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

criterion_seed42_test = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_seed42_test = optim.Adam(
    model_seed42_test.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("SEED 42 — ORIGINAL-CONDITIONS MODEL")
print("=" * 70)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_seed42_test.parameters()
        if p.requires_grad
    )
)

print("Learning rate: 1e-4")
print("Deterministic: False")
print("Benchmark: True")
print("=" * 70)

In [ ]:
# ============================================================
# Seed 42 — Stage 1 Training
# Original Conditions
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 3

best_macro_f1 = -1.0
best_epoch = 0

print("=" * 70)
print("SEED 42 — STAGE 1")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_seed42_test.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_seed42_test.zero_grad()

        outputs = model_seed42_test(images)

        loss = criterion_seed42_test(
            outputs,
            labels
        )

        loss.backward()

        optimizer_seed42_test.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total

    train_acc = (
        train_correct /
        train_total
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_seed42_test.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_seed42_test(images)

            loss = criterion_seed42_test(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Best result
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1
        best_epoch = epoch + 1

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Macro F1: {best_macro_f1:.4f}"
)

print(
    f"Best Epoch: {best_epoch}"
)

In [ ]:
# ============================================================
# Robustness — Seed 42
# Correct starting point: Best CLAHE checkpoint
# ============================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

SEED = 42

# ------------------------------------------------------------
# Set seed
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# ------------------------------------------------------------
# Build ResNet18
# ------------------------------------------------------------

model_seed42 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_seed42.fc = nn.Linear(
    model_seed42.fc.in_features,
    5
)

model_seed42 = model_seed42.to(device)


# ------------------------------------------------------------
# Load BEST D checkpoint
# ------------------------------------------------------------

checkpoint_d_seed42 = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_seed42.load_state_dict(
    checkpoint_d_seed42["model_state_dict"]
)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_seed42.parameters():
    param.requires_grad = False

for param in model_seed42.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

criterion_seed42 = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# Stage 1 optimizer
# ------------------------------------------------------------

optimizer_seed42_stage1 = optim.Adam(
    model_seed42.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("SEED 42 — CORRECT STAGE 1 STARTING POINT")
print("=" * 70)

print(
    "Loaded checkpoint:",
    "./best_resnet18_oa_clahe.pth"
)

print(
    "Starting Macro F1:",
    checkpoint_d_seed42["macro_f1"]
)

print(
    "Checkpoint epoch:",
    checkpoint_d_seed42["epoch"]
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_seed42.parameters()
        if p.requires_grad
    )
)

print("=" * 70)

In [ ]:
# ============================================================
# Seed 42 — Correct Stage 1 Fine-Tuning
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 3

best_macro_f1_seed42 = checkpoint_d_seed42["macro_f1"]
best_epoch_seed42 = checkpoint_d_seed42["epoch"]

BEST_SEED42_STAGE1 = (
    "./best_resnet18_oa_seed42_stage1_correct.pth"
)

print("=" * 70)
print("SEED 42 — STAGE 1")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_seed42.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_seed42_stage1.zero_grad()

        outputs = model_seed42(images)

        loss = criterion_seed42(
            outputs,
            labels
        )

        loss.backward()

        optimizer_seed42_stage1.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_seed42.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_seed42(images)

            loss = criterion_seed42(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_seed42:

        best_macro_f1_seed42 = macro_f1

        best_epoch_seed42 = (
            checkpoint_d_seed42["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "seed": SEED,
                "epoch": best_epoch_seed42,
                "model_state_dict":
                    model_seed42.state_dict(),
                "optimizer_state_dict":
                    optimizer_seed42_stage1.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_SEED42_STAGE1
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_seed42['macro_f1']:.4f}"
)

print(
    f"Best Seed 42 Stage 1 Macro F1: "
    f"{best_macro_f1_seed42:.4f}"
)

print(
    f"Best epoch: {best_epoch_seed42}"
)

print(
    f"Saved model: {BEST_SEED42_STAGE1}"
)

In [ ]:
# ============================================================
# Seed 42 — Stage 2
# Full ResNet18 Fine-Tuning
# ============================================================

# ------------------------------------------------------------
# Load best Stage 1 checkpoint
# ------------------------------------------------------------

checkpoint_seed42_stage1 = torch.load(
    BEST_SEED42_STAGE1,
    map_location=device,
    weights_only=False
)

model_seed42.load_state_dict(
    checkpoint_seed42_stage1["model_state_dict"]
)

model_seed42 = model_seed42.to(device)


# ------------------------------------------------------------
# Unfreeze entire model
# ------------------------------------------------------------

for param in model_seed42.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Stage 2 optimizer
# ------------------------------------------------------------

optimizer_seed42_stage2 = optim.Adam(
    [
        {
            "params": model_seed42.layer1.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_seed42.layer2.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_seed42.layer3.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_seed42.layer4.parameters(),
            "lr": 1e-5
        },
        {
            "params": model_seed42.fc.parameters(),
            "lr": 1e-5
        }
    ]
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in model_seed42.parameters()
    if p.requires_grad
)

print("=" * 70)
print("SEED 42 — STAGE 2 READY")
print("=" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_seed42_stage1['macro_f1']:.4f}"
)

print(
    f"Starting Val Acc: "
    f"{checkpoint_seed42_stage1['val_acc']:.4f}"
)

print(
    f"Stage 1 epoch: "
    f"{checkpoint_seed42_stage1['epoch']}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print("Learning rate: 1e-5")

print("=" * 70)

In [ ]:
# ============================================================
# Seed 42 — Stage 2 Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS_STAGE2 = 5

best_macro_f1_seed42_stage2 = (
    checkpoint_seed42_stage1["macro_f1"]
)

best_epoch_seed42_stage2 = (
    checkpoint_seed42_stage1["epoch"]
)

BEST_SEED42_STAGE2 = (
    "./best_resnet18_oa_seed42_stage2.pth"
)

print("=" * 70)
print("SEED 42 — STAGE 2 TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS_STAGE2):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_seed42.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_seed42_stage2.zero_grad()

        outputs = model_seed42(images)

        loss = criterion_seed42(
            outputs,
            labels
        )

        loss.backward()

        optimizer_seed42_stage2.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_seed42.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_seed42(images)

            loss = criterion_seed42(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_seed42_stage2:

        best_macro_f1_seed42_stage2 = macro_f1

        best_epoch_seed42_stage2 = (
            checkpoint_seed42_stage1["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "seed": SEED,
                "epoch": best_epoch_seed42_stage2,
                "model_state_dict":
                    model_seed42.state_dict(),
                "optimizer_state_dict":
                    optimizer_seed42_stage2.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
            },
            BEST_SEED42_STAGE2
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS_STAGE2}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_seed42_stage1['macro_f1']:.4f}"
)

print(
    f"Best Seed 42 Stage 2 Macro F1: "
    f"{best_macro_f1_seed42_stage2:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_seed42_stage2}"
)

print(
    f"Saved model: "
    f"{BEST_SEED42_STAGE2}"
)

In [ ]:
# ============================================================
# Experiment: Ordinal-Aware Loss
# Step 1 — Define and test the loss
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class OrdinalAwareLoss(nn.Module):

    def __init__(
        self,
        class_weights,
        ordinal_weight=0.5
    ):
        super().__init__()

        self.ce = nn.CrossEntropyLoss(
            weight=class_weights
        )

        self.ordinal_weight = ordinal_weight

        # KL grades: 0, 1, 2, 3, 4
        self.register_buffer(
            "class_values",
            torch.arange(
                5,
                dtype=torch.float32
            )
        )

    def forward(self, logits, targets):

        # ----------------------------------------
        # Standard weighted classification loss
        # ----------------------------------------

        ce_loss = self.ce(
            logits,
            targets
        )

        # ----------------------------------------
        # Convert logits to probabilities
        # ----------------------------------------

        probabilities = F.softmax(
            logits,
            dim=1
        )

        # ----------------------------------------
        # Expected KL grade
        # ----------------------------------------

        expected_grade = (
            probabilities *
            self.class_values
        ).sum(dim=1)

        # ----------------------------------------
        # Ordinal penalty
        # ----------------------------------------

        target_grade = targets.float()

        ordinal_loss = F.smooth_l1_loss(
            expected_grade,
            target_grade
        )

        # ----------------------------------------
        # Combined loss
        # ----------------------------------------

        total_loss = (
            ce_loss
            +
            self.ordinal_weight *
            ordinal_loss
        )

        return total_loss


# ------------------------------------------------------------
# Create loss
# ------------------------------------------------------------

ordinal_criterion = OrdinalAwareLoss(
    class_weights=class_weights_tensor,
    ordinal_weight=0.5
)


# ------------------------------------------------------------
# Simple sanity check
# ------------------------------------------------------------

dummy_logits = torch.randn(
    8,
    5,
    device=device
)

dummy_targets = torch.tensor(
    [0, 1, 2, 3, 4, 1, 2, 3],
    device=device
)

test_loss = ordinal_criterion(
    dummy_logits,
    dummy_targets
)

print("=" * 70)
print("ORDINAL-AWARE LOSS TEST")
print("=" * 70)

print("Loss value:", test_loss.item())

print(
    "Loss is finite:",
    torch.isfinite(test_loss).item()
)

print("=" * 70)

In [ ]:
ordinal_criterion = ordinal_criterion.to(device)

dummy_logits = torch.randn(
    8,
    5,
    device=device
)

dummy_targets = torch.tensor(
    [0, 1, 2, 3, 4, 1, 2, 3],
    device=device
)

test_loss = ordinal_criterion(
    dummy_logits,
    dummy_targets
)

print("=" * 70)
print("ORDINAL-AWARE LOSS TEST")
print("=" * 70)

print("Loss value:", test_loss.item())
print("Loss is finite:", torch.isfinite(test_loss).item())
print("Loss device:", test_loss.device)

print("=" * 70)

In [ ]:
# ============================================================
# Ordinal-Aware Experiment
# Step 2 — Build model from Best D checkpoint
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ------------------------------------------------------------
# Create fresh ResNet18
# ------------------------------------------------------------

model_ordinal = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_ordinal.fc = nn.Linear(
    model_ordinal.fc.in_features,
    5
)

model_ordinal = model_ordinal.to(device)


# ------------------------------------------------------------
# Load best D checkpoint
# ------------------------------------------------------------

checkpoint_d_ordinal = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_ordinal.load_state_dict(
    checkpoint_d_ordinal["model_state_dict"]
)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_ordinal.parameters():
    param.requires_grad = False

for param in model_ordinal.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Ordinal-aware loss
# ------------------------------------------------------------

criterion_ordinal = OrdinalAwareLoss(
    class_weights=class_weights_tensor,
    ordinal_weight=0.5
).to(device)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_ordinal = optim.Adam(
    model_ordinal.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("ORDINAL-AWARE EXPERIMENT READY")
print("=" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_ordinal['macro_f1']:.4f}"
)

print(
    f"Starting Val Acc: "
    f"{checkpoint_d_ordinal['val_acc']:.4f}"
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_ordinal.parameters()
        if p.requires_grad
    )
)

print("Ordinal weight: 0.5")
print("Learning rate: 1e-4")

print("=" * 70)

In [ ]:
# ============================================================
# Ordinal-Aware Experiment
# Step 3 — Train Classifier Head
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 3

best_macro_f1_ordinal = (
    checkpoint_d_ordinal["macro_f1"]
)

best_epoch_ordinal = (
    checkpoint_d_ordinal["epoch"]
)

BEST_ORDINAL_PATH = (
    "./best_resnet18_oa_ordinal.pth"
)

print("=" * 70)
print("ORDINAL-AWARE LOSS — STAGE 1 TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_ordinal.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_ordinal.zero_grad()

        outputs = model_ordinal(images)

        loss = criterion_ordinal(
            outputs,
            labels
        )

        loss.backward()

        optimizer_ordinal.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_ordinal.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_ordinal(images)

            loss = criterion_ordinal(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_ordinal:

        best_macro_f1_ordinal = macro_f1

        best_epoch_ordinal = (
            checkpoint_d_ordinal["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "epoch": best_epoch_ordinal,
                "model_state_dict":
                    model_ordinal.state_dict(),
                "optimizer_state_dict":
                    optimizer_ordinal.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "ordinal_weight": 0.5,
            },
            BEST_ORDINAL_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_ordinal['macro_f1']:.4f}"
)

print(
    f"Best Ordinal Macro F1: "
    f"{best_macro_f1_ordinal:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_ordinal}"
)

print(
    f"Saved model: "
    f"{BEST_ORDINAL_PATH}"
)

In [ ]:
# ============================================================
# Evaluate Best Stage 1 Model
# ============================================================

import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score
)

BEST_MODEL_PATH = "./best_resnet18_oa_stage1.pth"

checkpoint_best = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

# ------------------------------------------------------------
# Load best model
# ------------------------------------------------------------

model_ft.load_state_dict(
    checkpoint_best["model_state_dict"]
)

model_ft = model_ft.to(device)
model_ft.eval()


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader_clahe:

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model_ft(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

balanced_accuracy = balanced_accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

mae = mean_absolute_error(
    all_labels,
    all_predictions
)

qwk = cohen_kappa_score(
    all_labels,
    all_predictions,
    weights="quadratic"
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

print("=" * 70)
print("BEST STAGE 1 MODEL — VALIDATION")
print("=" * 70)

print(f"Checkpoint epoch   : {checkpoint_best['epoch']}")
print(f"Saved Macro F1     : {checkpoint_best['macro_f1']:.4f}")

print("\nOverall metrics:")
print(f"Accuracy           : {accuracy:.4f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.4f}")
print(f"Macro F1           : {macro_f1:.4f}")
print(f"MAE                : {mae:.4f}")
print(f"QWK                : {qwk:.4f}")


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print("Rows = True class")
print("Cols = Predicted class\n")

print(
    "             " +
    "  ".join(
        f"{name[:9]:>9}"
        for name in class_names
    )
)

for i, row in enumerate(cm):

    print(
        f"{class_names[i]:<10}" +
        "  ".join(
            f"{value:9d}"
            for value in row
        )
    )

In [ ]:
# ============================================================
# Experiment: Unweighted CrossEntropy
# Step 1 — Build model from Best D checkpoint
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ------------------------------------------------------------
# Create fresh ResNet18
# ------------------------------------------------------------

model_unweighted = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_unweighted.fc = nn.Linear(
    model_unweighted.fc.in_features,
    5
)

model_unweighted = model_unweighted.to(device)


# ------------------------------------------------------------
# Load Best D checkpoint
# ------------------------------------------------------------

checkpoint_d_unweighted = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_unweighted.load_state_dict(
    checkpoint_d_unweighted["model_state_dict"]
)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_unweighted.parameters():
    param.requires_grad = False

for param in model_unweighted.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# UNWEIGHTED CrossEntropy
# ------------------------------------------------------------

criterion_unweighted = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_unweighted = optim.Adam(
    model_unweighted.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("UNWEIGHTED CROSSENTROPY — EXPERIMENT READY")
print("=" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_unweighted['macro_f1']:.4f}"
)

print(
    f"Starting Val Acc: "
    f"{checkpoint_d_unweighted['val_acc']:.4f}"
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_unweighted.parameters()
        if p.requires_grad
    )
)

print("Loss: Unweighted CrossEntropy")
print("Learning rate: 1e-4")

print("=" * 70)

In [ ]:
# ============================================================
# Unweighted CrossEntropy — Stage 1 Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 3

best_macro_f1_unweighted = (
    checkpoint_d_unweighted["macro_f1"]
)

best_epoch_unweighted = (
    checkpoint_d_unweighted["epoch"]
)

BEST_UNWEIGHTED_PATH = (
    "./best_resnet18_oa_unweighted.pth"
)

print("=" * 70)
print("UNWEIGHTED CROSSENTROPY — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_unweighted.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_unweighted.zero_grad()

        outputs = model_unweighted(images)

        loss = criterion_unweighted(
            outputs,
            labels
        )

        loss.backward()

        optimizer_unweighted.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_unweighted.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_unweighted(images)

            loss = criterion_unweighted(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_unweighted:

        best_macro_f1_unweighted = macro_f1

        best_epoch_unweighted = (
            checkpoint_d_unweighted["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "epoch": best_epoch_unweighted,
                "model_state_dict":
                    model_unweighted.state_dict(),
                "optimizer_state_dict":
                    optimizer_unweighted.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "loss_type": "unweighted_cross_entropy",
            },
            BEST_UNWEIGHTED_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_unweighted['macro_f1']:.4f}"
)

print(
    f"Best Unweighted Macro F1: "
    f"{best_macro_f1_unweighted:.4f}"
)

print(
    f"Best epoch: {best_epoch_unweighted}"
)

print(
    f"Saved model: {BEST_UNWEIGHTED_PATH}"
)

In [ ]:
# ============================================================
# Focal Loss Experiment
# Step 1 — Define and test Focal Loss
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class WeightedFocalLoss(nn.Module):

    def __init__(
        self,
        class_weights,
        gamma=2.0
    ):
        super().__init__()

        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):

        # Standard cross-entropy for each sample
        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.class_weights,
            reduction="none"
        )

        # Probability assigned to the correct class
        pt = torch.exp(-ce_loss)

        # Focal weighting
        focal_loss = (
            (1 - pt) ** self.gamma
        ) * ce_loss

        return focal_loss.mean()


# ------------------------------------------------------------
# Create loss
# ------------------------------------------------------------

criterion_focal = WeightedFocalLoss(
    class_weights=class_weights_tensor,
    gamma=2.0
).to(device)


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

dummy_logits = torch.randn(
    8,
    5,
    device=device
)

dummy_targets = torch.tensor(
    [0, 1, 2, 3, 4, 1, 2, 3],
    device=device
)

test_focal_loss = criterion_focal(
    dummy_logits,
    dummy_targets
)


print("=" * 70)
print("FOCAL LOSS TEST")
print("=" * 70)

print(
    "Loss value:",
    test_focal_loss.item()
)

print(
    "Loss is finite:",
    torch.isfinite(test_focal_loss).item()
)

print(
    "Loss device:",
    test_focal_loss.device
)

print(
    "Gamma:",
    criterion_focal.gamma
)

print("=" * 70)

In [ ]:
# ============================================================
# Focal Loss Experiment
# Step 2 — Build model from Best D checkpoint
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ------------------------------------------------------------
# Create fresh ResNet18
# ------------------------------------------------------------

model_focal = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_focal.fc = nn.Linear(
    model_focal.fc.in_features,
    5
)

model_focal = model_focal.to(device)


# ------------------------------------------------------------
# Load Best D checkpoint
# ------------------------------------------------------------

checkpoint_d_focal = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_focal.load_state_dict(
    checkpoint_d_focal["model_state_dict"]
)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_focal.parameters():
    param.requires_grad = False

for param in model_focal.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Focal Loss
# ------------------------------------------------------------

criterion_focal = WeightedFocalLoss(
    class_weights=class_weights_tensor,
    gamma=2.0
).to(device)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_focal = optim.Adam(
    model_focal.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("FOCAL LOSS EXPERIMENT READY")
print("=" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_focal['macro_f1']:.4f}"
)

print(
    f"Starting Val Acc: "
    f"{checkpoint_d_focal['val_acc']:.4f}"
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_focal.parameters()
        if p.requires_grad
    )
)

print("Loss: Weighted Focal Loss")
print("Gamma: 2.0")
print("Learning rate: 1e-4")

print("=" * 70)

In [ ]:
# ============================================================
# Weighted Focal Loss — Stage 1 Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_focal = (
    checkpoint_d_focal["macro_f1"]
)

best_epoch_focal = (
    checkpoint_d_focal["epoch"]
)

BEST_FOCAL_PATH = (
    "./best_resnet18_oa_focal.pth"
)

print("=" * 70)
print("WEIGHTED FOCAL LOSS — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_focal.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_clahe:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_focal.zero_grad()

        outputs = model_focal(images)

        loss = criterion_focal(
            outputs,
            labels
        )

        loss.backward()

        optimizer_focal.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_focal.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_focal(images)

            loss = criterion_focal(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_focal:

        best_macro_f1_focal = macro_f1

        best_epoch_focal = (
            checkpoint_d_focal["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "epoch": best_epoch_focal,
                "model_state_dict":
                    model_focal.state_dict(),
                "optimizer_state_dict":
                    optimizer_focal.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "loss_type": "weighted_focal",
                "gamma": 2.0,
            },
            BEST_FOCAL_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_focal['macro_f1']:.4f}"
)

print(
    f"Best Focal Macro F1: "
    f"{best_macro_f1_focal:.4f}"
)

print(
    f"Best epoch: {best_epoch_focal}"
)

print(
    f"Saved model: {BEST_FOCAL_PATH}"
)

In [ ]:
# ============================================================
# Experiment: WeightedRandomSampler
# Step 1 — Build balanced training DataLoader
# ============================================================

from torch.utils.data import WeightedRandomSampler, DataLoader
import numpy as np

# ------------------------------------------------------------
# Get labels from training dataset
# ------------------------------------------------------------

train_targets = np.array(
    train_dataset_clahe.targets
)

# Count samples in each class
class_counts_sampler = np.bincount(
    train_targets,
    minlength=5
)

# Weight for each class
class_sample_weights = (
    1.0 / class_counts_sampler
)

# Weight assigned to each individual sample
sample_weights = class_sample_weights[
    train_targets
]

sample_weights = torch.DoubleTensor(
    sample_weights
)


# ------------------------------------------------------------
# Create sampler
# ------------------------------------------------------------

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


# ------------------------------------------------------------
# New training DataLoader
# ------------------------------------------------------------

train_loader_sampler = DataLoader(
    train_dataset_clahe,
    batch_size=32,
    sampler=weighted_sampler,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Check class distribution
# ------------------------------------------------------------

print("=" * 70)
print("WEIGHTED RANDOM SAMPLER READY")
print("=" * 70)

print("Original class counts:")
for i, count in enumerate(class_counts_sampler):
    print(f"Class {i}: {count}")

print("\nClass sampling weights:")
for i, weight in enumerate(class_sample_weights):
    print(f"Class {i}: {weight:.8f}")

print("\nNumber of samples per epoch:")
print(len(train_loader_sampler.dataset))

print("\nBatch size:")
print(train_loader_sampler.batch_size)

print("=" * 70)

In [ ]:
# ============================================================
# WeightedRandomSampler Experiment
# Step 2 — Model + Unweighted CrossEntropy
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ------------------------------------------------------------
# Fresh ResNet18
# ------------------------------------------------------------

model_sampler = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_sampler.fc = nn.Linear(
    model_sampler.fc.in_features,
    5
)

model_sampler = model_sampler.to(device)


# ------------------------------------------------------------
# Load Best D checkpoint
# ------------------------------------------------------------

checkpoint_d_sampler = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_sampler.load_state_dict(
    checkpoint_d_sampler["model_state_dict"]
)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_sampler.parameters():
    param.requires_grad = False

for param in model_sampler.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Unweighted CrossEntropy
# ------------------------------------------------------------

criterion_sampler = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_sampler = optim.Adam(
    model_sampler.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("SAMPLER EXPERIMENT READY")
print("=" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_sampler['macro_f1']:.4f}"
)

print(
    f"Starting Val Acc: "
    f"{checkpoint_d_sampler['val_acc']:.4f}"
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_sampler.parameters()
        if p.requires_grad
    )
)

print("Sampler: WeightedRandomSampler")
print("Loss: Unweighted CrossEntropy")
print("Learning rate: 1e-4")

print("=" * 70)

In [ ]:
# ============================================================
# WeightedRandomSampler + Unweighted CE — Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_sampler = (
    checkpoint_d_sampler["macro_f1"]
)

best_epoch_sampler = (
    checkpoint_d_sampler["epoch"]
)

BEST_SAMPLER_PATH = (
    "./best_resnet18_oa_sampler.pth"
)

print("=" * 70)
print("SAMPLER EXPERIMENT — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_sampler.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_sampler:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_sampler.zero_grad()

        outputs = model_sampler(images)

        loss = criterion_sampler(
            outputs,
            labels
        )

        loss.backward()

        optimizer_sampler.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_sampler.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_sampler(images)

            loss = criterion_sampler(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_sampler:

        best_macro_f1_sampler = macro_f1

        best_epoch_sampler = (
            checkpoint_d_sampler["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "epoch": best_epoch_sampler,
                "model_state_dict":
                    model_sampler.state_dict(),
                "optimizer_state_dict":
                    optimizer_sampler.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "loss_type": "unweighted_cross_entropy",
                "sampler": "weighted_random_sampler",
            },
            BEST_SAMPLER_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_sampler['macro_f1']:.4f}"
)

print(
    f"Best Sampler Macro F1: "
    f"{best_macro_f1_sampler:.4f}"
)

print(
    f"Best epoch: {best_epoch_sampler}"
)

print(
    f"Saved model: {BEST_SAMPLER_PATH}"
)

In [ ]:
# ============================================================
# Evaluate Best Sampler Model — Validation
# ============================================================

import torch

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score
)

BEST_SAMPLER_PATH = "./best_resnet18_oa_sampler.pth"

checkpoint_sampler_best = torch.load(
    BEST_SAMPLER_PATH,
    map_location=device,
    weights_only=False
)

# ------------------------------------------------------------
# Load best model
# ------------------------------------------------------------

model_sampler.load_state_dict(
    checkpoint_sampler_best["model_state_dict"]
)

model_sampler = model_sampler.to(device)
model_sampler.eval()


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader_clahe:

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model_sampler(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

balanced_accuracy = balanced_accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

mae = mean_absolute_error(
    all_labels,
    all_predictions
)

qwk = cohen_kappa_score(
    all_labels,
    all_predictions,
    weights="quadratic"
)


class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 70)
print("BEST SAMPLER MODEL — VALIDATION")
print("=" * 70)

print(
    f"Checkpoint epoch   : "
    f"{checkpoint_sampler_best['epoch']}"
)

print(
    f"Saved Macro F1     : "
    f"{checkpoint_sampler_best['macro_f1']:.4f}"
)

print("\nOverall metrics:")

print(
    f"Accuracy           : "
    f"{accuracy:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{balanced_accuracy:.4f}"
)

print(
    f"Macro F1           : "
    f"{macro_f1:.4f}"
)

print(
    f"MAE                : "
    f"{mae:.4f}"
)

print(
    f"QWK                : "
    f"{qwk:.4f}"
)


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print("Rows = True class")
print("Cols = Predicted class\n")

print(
    "             " +
    "  ".join(
        f"{name[:9]:>9}"
        for name in class_names
    )
)

for i, row in enumerate(cm):

    print(
        f"{class_names[i]:<10}" +
        "  ".join(
            f"{value:9d}"
            for value in row
        )
    )

In [ ]:
# ============================================================
# Experiment: WeightedRandomSampler + Weighted CrossEntropy
# Step 1 — Build model
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ------------------------------------------------------------
# Fresh ResNet18
# ------------------------------------------------------------

model_sampler_weighted = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_sampler_weighted.fc = nn.Linear(
    model_sampler_weighted.fc.in_features,
    5
)

model_sampler_weighted = model_sampler_weighted.to(device)


# ------------------------------------------------------------
# Load Best D checkpoint
# ------------------------------------------------------------

checkpoint_d_sampler_weighted = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_sampler_weighted.load_state_dict(
    checkpoint_d_sampler_weighted["model_state_dict"]
)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_sampler_weighted.parameters():
    param.requires_grad = False

for param in model_sampler_weighted.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Weighted CrossEntropy
# ------------------------------------------------------------

criterion_sampler_weighted = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_sampler_weighted = optim.Adam(
    model_sampler_weighted.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("SAMPLER + WEIGHTED CE — EXPERIMENT READY")
print("=" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_sampler_weighted['macro_f1']:.4f}"
)

print(
    f"Starting Val Acc: "
    f"{checkpoint_d_sampler_weighted['val_acc']:.4f}"
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_sampler_weighted.parameters()
        if p.requires_grad
    )
)

print("Sampler: WeightedRandomSampler")
print("Loss: Weighted CrossEntropy")
print("Learning rate: 1e-4")

print("=" * 70)

In [ ]:
# ============================================================
# WeightedRandomSampler + Weighted CE — Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_sampler_weighted = (
    checkpoint_d_sampler_weighted["macro_f1"]
)

best_epoch_sampler_weighted = (
    checkpoint_d_sampler_weighted["epoch"]
)

BEST_SAMPLER_WEIGHTED_PATH = (
    "./best_resnet18_oa_sampler_weighted.pth"
)

print("=" * 70)
print("SAMPLER + WEIGHTED CE — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_sampler_weighted.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_sampler:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_sampler_weighted.zero_grad()

        outputs = model_sampler_weighted(images)

        loss = criterion_sampler_weighted(
            outputs,
            labels
        )

        loss.backward()

        optimizer_sampler_weighted.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_sampler_weighted.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_clahe:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_sampler_weighted(images)

            loss = criterion_sampler_weighted(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_sampler_weighted:

        best_macro_f1_sampler_weighted = macro_f1

        best_epoch_sampler_weighted = (
            checkpoint_d_sampler_weighted["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "epoch": best_epoch_sampler_weighted,
                "model_state_dict":
                    model_sampler_weighted.state_dict(),
                "optimizer_state_dict":
                    optimizer_sampler_weighted.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "loss_type": "weighted_cross_entropy",
                "sampler": "weighted_random_sampler",
            },
            BEST_SAMPLER_WEIGHTED_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)

print(
    f"Starting Macro F1: "
    f"{checkpoint_d_sampler_weighted['macro_f1']:.4f}"
)

print(
    f"Best Sampler + Weighted CE Macro F1: "
    f"{best_macro_f1_sampler_weighted:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_sampler_weighted}"
)

print(
    f"Saved model: "
    f"{BEST_SAMPLER_WEIGHTED_PATH}"
)

In [ ]:
# ============================================================
# Error Analysis — KL0 / KL1 / KL2
# Show misclassified validation images
# ============================================================

import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# ------------------------------------------------------------
# Load best sampler model
# ------------------------------------------------------------

BEST_SAMPLER_PATH = "./best_resnet18_oa_sampler.pth"

checkpoint_error = torch.load(
    BEST_SAMPLER_PATH,
    map_location=device,
    weights_only=False
)

model_sampler.load_state_dict(
    checkpoint_error["model_state_dict"]
)

model_sampler = model_sampler.to(device)
model_sampler.eval()


class_names = {
    0: "Normal",
    1: "Doubtful",
    2: "Mild",
    3: "Moderate",
    4: "Severe"
}


# ------------------------------------------------------------
# Collect errors involving KL0 / KL1 / KL2
# ------------------------------------------------------------

error_samples = []

with torch.no_grad():

    for images, labels in val_loader_clahe:

        images_gpu = images.to(
            device,
            non_blocking=True
        )

        outputs = model_sampler(
            images_gpu
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        ).cpu().numpy()

        labels_np = labels.numpy()

        for i in range(len(labels_np)):

            true_label = int(labels_np[i])
            pred_label = int(predictions[i])

            # Only errors among Normal / Doubtful / Mild
            if (
                true_label in [0, 1, 2]
                and pred_label in [0, 1, 2]
                and true_label != pred_label
            ):

                error_samples.append(
                    {
                        "image": images[i],
                        "true": true_label,
                        "pred": pred_label
                    }
                )


# ------------------------------------------------------------
# Print total number of relevant errors
# ------------------------------------------------------------

print("=" * 70)
print("KL0 / KL1 / KL2 ERROR ANALYSIS")
print("=" * 70)

print(
    f"Total relevant errors: "
    f"{len(error_samples)}"
)


# ------------------------------------------------------------
# Randomly select up to 20 errors
# ------------------------------------------------------------

NUM_SHOW = min(
    20,
    len(error_samples)
)

random.seed(42)

selected_samples = random.sample(
    error_samples,
    NUM_SHOW
)


# ------------------------------------------------------------
# Display images
# ------------------------------------------------------------

fig, axes = plt.subplots(
    4,
    5,
    figsize=(16, 13)
)

axes = axes.flatten()

for ax in axes:
    ax.axis("off")


for ax, sample in zip(
    axes,
    selected_samples
):

    image = sample["image"]

    # First channel is enough because all 3 channels
    # contain the same grayscale image
    image_np = image[0].numpy()

    # Normalize only for visualization
    image_np = (
        image_np - image_np.min()
    ) / (
        image_np.max() -
        image_np.min() +
        1e-8
    )

    ax.imshow(
        image_np,
        cmap="gray"
    )

    ax.set_title(
        f"True: {class_names[sample['true']]}\n"
        f"Pred: {class_names[sample['pred']]}",
        fontsize=10
    )

    ax.axis("off")


plt.suptitle(
    "Misclassified Validation Samples — KL0 / KL1 / KL2",
    fontsize=16
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Experiment: Percentile Intensity Normalization + CLAHE
# Step 1 — New preprocessing
# ============================================================

import numpy as np
import cv2

from PIL import Image
from torchvision import transforms


class PercentileNormalize:
    """
    Normalize image intensity using percentile clipping.

    The lowest 1% and highest 1% of pixel intensities
    are clipped, then the image is rescaled to [0, 255].
    """

    def __init__(self, lower=1, upper=99):
        self.lower = lower
        self.upper = upper

    def __call__(self, img):

        img = np.array(img)

        # Convert RGB to grayscale if necessary
        if len(img.shape) == 3:
            img = cv2.cvtColor(
                img,
                cv2.COLOR_RGB2GRAY
            )

        img = img.astype(np.float32)

        # Percentile limits
        low = np.percentile(
            img,
            self.lower
        )

        high = np.percentile(
            img,
            self.upper
        )

        # Prevent division by zero
        if high <= low:
            return Image.fromarray(
                img.astype(np.uint8)
            )

        # Clip and rescale
        img = np.clip(
            img,
            low,
            high
        )

        img = (
            (img - low) /
            (high - low)
            * 255.0
        )

        img = img.astype(np.uint8)

        return Image.fromarray(img)


# ------------------------------------------------------------
# New training transform
# ------------------------------------------------------------

train_transform_contrast = transforms.Compose([
    transforms.Resize((224, 224)),

    # NEW STEP
    PercentileNormalize(
        lower=1,
        upper=99
    ),

    # Existing CLAHE
    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# New validation transform
# ------------------------------------------------------------

eval_transform_contrast = transforms.Compose([
    transforms.Resize((224, 224)),

    # NEW STEP
    PercentileNormalize(
        lower=1,
        upper=99
    ),

    # Existing CLAHE
    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


print("=" * 70)
print("CONTRAST NORMALIZATION TRANSFORMS READY")
print("=" * 70)

print("New step: Percentile normalization")
print("Percentiles: 1% - 99%")
print("CLAHE: clipLimit=2.0, grid=(8,8)")
print("Input size: 224x224")

print("=" * 70)

In [ ]:
# ============================================================
# Contrast Normalization Experiment
# Step 2 — Build datasets and DataLoaders
# ============================================================

from torchvision import datasets
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset_contrast = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform_contrast
)

val_dataset_contrast = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform_contrast
)

test_dataset_contrast = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform_contrast
)


# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader_contrast = DataLoader(
    train_dataset_contrast,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_contrast = DataLoader(
    val_dataset_contrast,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_contrast = DataLoader(
    test_dataset_contrast,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("CONTRAST NORMALIZATION DATA PIPELINE READY")
print("=" * 70)

print(
    "Train samples:",
    len(train_dataset_contrast)
)

print(
    "Validation samples:",
    len(val_dataset_contrast)
)

print(
    "Test samples:",
    len(test_dataset_contrast)
)

print(
    "\nClass mapping:",
    train_dataset_contrast.class_to_idx
)

# Check one batch
images, labels = next(
    iter(train_loader_contrast)
)

print(
    "\nBatch shape:",
    images.shape
)

print(
    "Batch labels:",
    labels[:10].tolist()
)

print(
    "Image range:",
    images.min().item(),
    "to",
    images.max().item()
)

print("=" * 70)

In [ ]:
# ============================================================
# Contrast Normalization Experiment
# Step 3 — Build fresh ResNet18
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models

# ------------------------------------------------------------
# Fresh pretrained ResNet18
# ------------------------------------------------------------

model_contrast = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_contrast.fc = nn.Linear(
    model_contrast.fc.in_features,
    5
)

model_contrast = model_contrast.to(device)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_contrast.parameters():
    param.requires_grad = False

for param in model_contrast.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Weighted CrossEntropy
# ------------------------------------------------------------

criterion_contrast = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_contrast = optim.Adam(
    model_contrast.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

trainable_params_contrast = sum(
    p.numel()
    for p in model_contrast.parameters()
    if p.requires_grad
)

total_params_contrast = sum(
    p.numel()
    for p in model_contrast.parameters()
)

print("=" * 70)
print("CONTRAST NORMALIZATION MODEL READY")
print("=" * 70)

print(
    f"Total parameters     : "
    f"{total_params_contrast:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params_contrast:,}"
)

print("Loss: Weighted CrossEntropy")
print("Learning rate: 1e-4")
print("Backbone: Frozen")
print("=" * 70)

In [ ]:
# ============================================================
# Contrast Normalization — Stage 1 Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_contrast = -1.0
best_epoch_contrast = 0

BEST_CONTRAST_PATH = (
    "./best_resnet18_oa_contrast_norm.pth"
)

print("=" * 70)
print("CONTRAST NORMALIZATION — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_contrast.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_contrast:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_contrast.zero_grad()

        outputs = model_contrast(images)

        loss = criterion_contrast(
            outputs,
            labels
        )

        loss.backward()

        optimizer_contrast.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total

    train_acc = (
        train_correct /
        train_total
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_contrast.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_contrast:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_contrast(images)

            loss = criterion_contrast(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_contrast:

        best_macro_f1_contrast = macro_f1
        best_epoch_contrast = epoch + 1

        torch.save(
            {
                "epoch": best_epoch_contrast,
                "model_state_dict":
                    model_contrast.state_dict(),
                "optimizer_state_dict":
                    optimizer_contrast.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "preprocessing":
                    "percentile_1_99_plus_clahe",
                "loss_type":
                    "weighted_cross_entropy",
            },
            BEST_CONTRAST_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Contrast Macro F1: "
    f"{best_macro_f1_contrast:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_contrast}"
)

print(
    f"Saved model: "
    f"{BEST_CONTRAST_PATH}"
)

In [ ]:
# ============================================================
# Experiment: Higher Resolution 224 -> 384
# Step 1 — Build 384x384 Data Pipeline
# ============================================================

from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Train transform — same as best pipeline, only resolution
# changes from 224 to 384
# ------------------------------------------------------------

train_transform_384 = transforms.Compose([
    transforms.Resize((384, 384)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Validation transform
# ------------------------------------------------------------

eval_transform_384 = transforms.Compose([
    transforms.Resize((384, 384)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Datasets
# ------------------------------------------------------------

train_dataset_384 = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform_384
)

val_dataset_384 = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform_384
)


# ------------------------------------------------------------
# Balanced sampler
# Same strategy as our current best model
# ------------------------------------------------------------

train_targets_384 = np.array(
    train_dataset_384.targets
)

class_counts_384 = np.bincount(
    train_targets_384,
    minlength=5
)

class_sample_weights_384 = (
    1.0 / class_counts_384
)

sample_weights_384 = (
    class_sample_weights_384[
        train_targets_384
    ]
)

sample_weights_384 = torch.DoubleTensor(
    sample_weights_384
)

weighted_sampler_384 = WeightedRandomSampler(
    weights=sample_weights_384,
    num_samples=len(sample_weights_384),
    replacement=True
)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

# Batch 16 to reduce T4 memory usage at 384x384
train_loader_384 = DataLoader(
    train_dataset_384,
    batch_size=16,
    sampler=weighted_sampler_384,
    num_workers=2,
    pin_memory=True
)

val_loader_384 = DataLoader(
    val_dataset_384,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

images_384, labels_384 = next(
    iter(train_loader_384)
)

print("=" * 70)
print("384x384 DATA PIPELINE READY")
print("=" * 70)

print(
    "Train samples:",
    len(train_dataset_384)
)

print(
    "Validation samples:",
    len(val_dataset_384)
)

print(
    "Batch shape:",
    images_384.shape
)

print(
    "Batch size:",
    train_loader_384.batch_size
)

print(
    "Class mapping:",
    train_dataset_384.class_to_idx
)

print(
    "Image range:",
    images_384.min().item(),
    "to",
    images_384.max().item()
)

print("=" * 70)

In [ ]:
# ============================================================
# 384x384 Experiment
# Step 2 — Build ResNet18
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ------------------------------------------------------------
# Fresh pretrained ResNet18
# ------------------------------------------------------------

model_384 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_384.fc = nn.Linear(
    model_384.fc.in_features,
    5
)

model_384 = model_384.to(device)


# ------------------------------------------------------------
# Freeze backbone
# ------------------------------------------------------------

for param in model_384.parameters():
    param.requires_grad = False

for param in model_384.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Unweighted CrossEntropy
# Same loss as current best model
# ------------------------------------------------------------

criterion_384 = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_384 = optim.Adam(
    model_384.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

total_params_384 = sum(
    p.numel()
    for p in model_384.parameters()
)

trainable_params_384 = sum(
    p.numel()
    for p in model_384.parameters()
    if p.requires_grad
)

print("=" * 70)
print("384x384 MODEL READY")
print("=" * 70)

print(
    f"Total parameters     : {total_params_384:,}"
)

print(
    f"Trainable parameters : {trainable_params_384:,}"
)

print("Resolution: 384x384")
print("Sampler: WeightedRandomSampler")
print("Loss: Unweighted CrossEntropy")
print("Learning rate: 1e-4")
print("Backbone: Frozen")

print("=" * 70)

In [ ]:
# ============================================================
# 384x384 Experiment — Stage 1 Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_384 = -1.0
best_epoch_384 = 0

BEST_384_PATH = (
    "./best_resnet18_oa_384.pth"
)

print("=" * 70)
print("384x384 — TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_384.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_384:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_384.zero_grad()

        outputs = model_384(images)

        loss = criterion_384(
            outputs,
            labels
        )

        loss.backward()

        optimizer_384.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_384.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_384:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_384(images)

            loss = criterion_384(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_384:

        best_macro_f1_384 = macro_f1
        best_epoch_384 = epoch + 1

        torch.save(
            {
                "epoch": best_epoch_384,
                "model_state_dict":
                    model_384.state_dict(),
                "optimizer_state_dict":
                    optimizer_384.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "resolution": "384x384",
                "sampler": "weighted_random_sampler",
                "loss_type": "unweighted_cross_entropy",
            },
            BEST_384_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best 384x384 Macro F1: "
    f"{best_macro_f1_384:.4f}"
)

print(
    f"Best epoch: {best_epoch_384}"
)

print(
    f"Saved model: {BEST_384_PATH}"
)

In [ ]:
# ============================================================
# 384x384 — CORRECT Experiment
# Step 2 — Start from the SAME Best D checkpoint
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# ------------------------------------------------------------
# Fresh ResNet18
# ------------------------------------------------------------

model_384 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_384.fc = nn.Linear(
    model_384.fc.in_features,
    5
)

model_384 = model_384.to(device)


# ------------------------------------------------------------
# Load the SAME Best D checkpoint used for 224 experiment
# ------------------------------------------------------------

checkpoint_d_384 = torch.load(
    "./best_resnet18_oa_clahe.pth",
    map_location=device,
    weights_only=False
)

model_384.load_state_dict(
    checkpoint_d_384["model_state_dict"]
)


# ------------------------------------------------------------
# Freeze backbone — same as Stage 1
# ------------------------------------------------------------

for param in model_384.parameters():
    param.requires_grad = False

for param in model_384.fc.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Same loss as winning 224 experiment
# ------------------------------------------------------------

criterion_384 = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# Same optimizer
# ------------------------------------------------------------

optimizer_384 = optim.Adam(
    model_384.fc.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("384x384 — CORRECT COMPARISON SETUP")
print("=" * 70)

print(
    f"Starting D Macro F1: "
    f"{checkpoint_d_384['macro_f1']:.4f}"
)

print(
    f"Starting D Val Acc: "
    f"{checkpoint_d_384['val_acc']:.4f}"
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_384.parameters()
        if p.requires_grad
    )
)

print("Resolution: 384x384")
print("Sampler: WeightedRandomSampler")
print("Loss: Unweighted CrossEntropy")
print("Learning rate: 1e-4")
print("Starting checkpoint: Best D")

print("=" * 70)

In [ ]:
# ============================================================
# 384x384 — Correct Resolution Ablation
# Stage 1 Training
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_384_correct = (
    checkpoint_d_384["macro_f1"]
)

best_epoch_384_correct = (
    checkpoint_d_384["epoch"]
)

BEST_384_CORRECT_PATH = (
    "./best_resnet18_oa_384_correct.pth"
)

print("=" * 70)
print("384x384 — CORRECT TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model_384.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_384:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_384.zero_grad()

        outputs = model_384(images)

        loss = criterion_384(
            outputs,
            labels
        )

        loss.backward()

        optimizer_384.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model_384.eval()

    val_loss = 0.0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader_384:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_384(images)

            loss = criterion_384(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = sum(
        p == y
        for p, y in zip(
            all_predictions,
            all_labels
        )
    ) / val_total

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_384_correct:

        best_macro_f1_384_correct = macro_f1

        best_epoch_384_correct = (
            checkpoint_d_384["epoch"]
            + epoch
            + 1
        )

        torch.save(
            {
                "epoch": best_epoch_384_correct,
                "model_state_dict":
                    model_384.state_dict(),
                "optimizer_state_dict":
                    optimizer_384.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "resolution": "384x384",
                "sampler": "weighted_random_sampler",
                "loss_type": "unweighted_cross_entropy",
                "starting_checkpoint": "best_D",
            },
            BEST_384_CORRECT_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )

print("-" * 70)

print(
    f"Starting D Macro F1: "
    f"{checkpoint_d_384['macro_f1']:.4f}"
)

print(
    f"Best 384x384 Macro F1: "
    f"{best_macro_f1_384_correct:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch_384_correct}"
)

print(
    f"Saved model: "
    f"{BEST_384_CORRECT_PATH}"
)

In [ ]:
# ============================================================
# Classical Feature Extraction
# Step 1 — Intensity + GLCM + LBP
# ============================================================

import os
import numpy as np
import pandas as pd
import cv2

from PIL import Image
from skimage.feature import (
    graycomatrix,
    graycoprops,
    local_binary_pattern
)

# ------------------------------------------------------------
# Feature extraction function
# ------------------------------------------------------------

def extract_classical_features(image_path):

    # Load grayscale image
    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    # Resize to a fixed size
    image = cv2.resize(
        image,
        (224, 224)
    )

    # --------------------------------------------------------
    # Intensity features
    # --------------------------------------------------------

    mean_intensity = np.mean(image)
    std_intensity = np.std(image)

    min_intensity = np.min(image)
    max_intensity = np.max(image)

    # Percentiles
    p10 = np.percentile(image, 10)
    p25 = np.percentile(image, 25)
    p50 = np.percentile(image, 50)
    p75 = np.percentile(image, 75)
    p90 = np.percentile(image, 90)

    # --------------------------------------------------------
    # Entropy
    # --------------------------------------------------------

    histogram = cv2.calcHist(
        [image],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    histogram = histogram / (
        histogram.sum() + 1e-8
    )

    entropy = -np.sum(
        histogram *
        np.log2(histogram + 1e-10)
    )

    # --------------------------------------------------------
    # GLCM
    # --------------------------------------------------------

    # Reduce gray levels from 256 -> 32
    # This makes GLCM much smaller and more stable.
    image_32 = (
        image / 8
    ).astype(np.uint8)

    glcm = graycomatrix(
        image_32,
        distances=[1, 2],
        angles=[
            0,
            np.pi / 4,
            np.pi / 2,
            3 * np.pi / 4
        ],
        levels=32,
        symmetric=True,
        normed=True
    )

    glcm_features = {}

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation"
    ]:

        values = graycoprops(
            glcm,
            prop
        )

        glcm_features[
            f"glcm_{prop}"
        ] = values.mean()

    # --------------------------------------------------------
    # LBP
    # --------------------------------------------------------

    lbp = local_binary_pattern(
        image,
        P=8,
        R=1,
        method="uniform"
    )

    # Histogram of LBP patterns
    n_bins = 10

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(
            0,
            n_bins + 1
        ),
        range=(0, n_bins)
    )

    lbp_hist = lbp_hist.astype(
        np.float32
    )

    lbp_hist /= (
        lbp_hist.sum() + 1e-8
    )

    # --------------------------------------------------------
    # Create feature dictionary
    # --------------------------------------------------------

    features = {
        "mean_intensity": mean_intensity,
        "std_intensity": std_intensity,
        "min_intensity": min_intensity,
        "max_intensity": max_intensity,

        "p10": p10,
        "p25": p25,
        "p50": p50,
        "p75": p75,
        "p90": p90,

        "entropy": entropy,
    }

    # Add GLCM
    features.update(
        glcm_features
    )

    # Add LBP histogram
    for i, value in enumerate(lbp_hist):
        features[
            f"lbp_{i}"
        ] = value

    return features


# ------------------------------------------------------------
# Test on one image
# ------------------------------------------------------------

sample_path = train_dataset_384.samples[0][0]

sample_features = extract_classical_features(
    sample_path
)

print("=" * 70)
print("CLASSICAL FEATURE EXTRACTION TEST")
print("=" * 70)

print(
    "Image:",
    os.path.basename(sample_path)
)

print(
    "Number of features:",
    len(sample_features)
)

print("\nExtracted features:")

for name, value in sample_features.items():
    print(
        f"{name:<25} {value:.6f}"
    )

print("=" * 70)

In [ ]:
# ============================================================
# Classical Features
# Step 2 — Extract features for Train + Validation
# ============================================================

from tqdm.auto import tqdm

def build_feature_dataframe(dataset, split_name):

    rows = []

    for image_path, label in tqdm(
        dataset.samples,
        desc=f"Extracting {split_name}"
    ):

        features = extract_classical_features(
            image_path
        )

        features["label"] = label
        features["image_path"] = image_path

        rows.append(features)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Extract TRAIN features
# ------------------------------------------------------------

train_features_df = build_feature_dataframe(
    train_dataset_384,
    "TRAIN"
)


# ------------------------------------------------------------
# Extract VALIDATION features
# ------------------------------------------------------------

val_features_df = build_feature_dataframe(
    val_dataset_384,
    "VALIDATION"
)


# ------------------------------------------------------------
# Check results
# ------------------------------------------------------------

print("=" * 70)
print("FEATURE DATASETS READY")
print("=" * 70)

print(
    "Train shape:",
    train_features_df.shape
)

print(
    "Validation shape:",
    val_features_df.shape
)

print("\nTrain class distribution:")

print(
    train_features_df["label"]
    .value_counts()
    .sort_index()
)

print("\nValidation class distribution:")

print(
    val_features_df["label"]
    .value_counts()
    .sort_index()
)

print("\nFeature columns:")

print(
    train_features_df.columns.tolist()
)

print("=" * 70)

In [ ]:
# ============================================================
# Classical Feature Analysis
# Step 3 — Feature Separability
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import kruskal
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Feature columns
# ------------------------------------------------------------

feature_columns = [
    col
    for col in train_features_df.columns
    if col not in ["label", "image_path"]
]

X_train_features = train_features_df[
    feature_columns
].copy()

y_train_features = train_features_df[
    "label"
].values


# ------------------------------------------------------------
# 1. Kruskal-Wallis test
# ------------------------------------------------------------

kruskal_results = []

for feature in feature_columns:

    groups = [
        train_features_df.loc[
            train_features_df["label"] == cls,
            feature
        ].values
        for cls in range(5)
    ]

    try:
        statistic, p_value = kruskal(
            *groups
        )
    except Exception:
        statistic = np.nan
        p_value = np.nan

    kruskal_results.append({
        "feature": feature,
        "H_statistic": statistic,
        "p_value": p_value
    })


kruskal_df = pd.DataFrame(
    kruskal_results
)

kruskal_df = kruskal_df.sort_values(
    "p_value"
)


# ------------------------------------------------------------
# 2. Mutual Information
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X_train_features
)

mi_scores = mutual_info_classif(
    X_scaled,
    y_train_features,
    random_state=42
)

mi_df = pd.DataFrame({
    "feature": feature_columns,
    "mutual_information": mi_scores
})

mi_df = mi_df.sort_values(
    "mutual_information",
    ascending=False
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 70)
print("FEATURE SEPARABILITY ANALYSIS")
print("=" * 70)


print("\nTop features by Kruskal-Wallis p-value:")
print("-" * 70)

print(
    kruskal_df.head(15).to_string(
        index=False
    )
)


print("\nTop features by Mutual Information:")
print("-" * 70)

print(
    mi_df.head(15).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 3. Per-class means for top features
# ------------------------------------------------------------

top_features = mi_df.head(
    8
)["feature"].tolist()

print("\n" + "=" * 70)
print("CLASS MEANS — TOP MUTUAL INFORMATION FEATURES")
print("=" * 70)

class_means = (
    train_features_df
    .groupby("label")[top_features]
    .mean()
)

class_means.index = [
    "KL0_Normal",
    "KL1_Doubtful",
    "KL2_Mild",
    "KL3_Moderate",
    "KL4_Severe"
]

print(
    class_means.to_string()
)

print("=" * 70)

In [ ]:
# ============================================================
# Rebuild Classical Feature Tables on CPU
# Step 1 — Extract + Save
# ============================================================

import os
import numpy as np
import pandas as pd
import cv2

from tqdm.auto import tqdm
from skimage.feature import (
    graycomatrix,
    graycoprops,
    local_binary_pattern
)


# ------------------------------------------------------------
# Feature extraction function
# ------------------------------------------------------------

def extract_classical_features(image_path):

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    image = cv2.resize(
        image,
        (224, 224)
    )

    # -----------------------------
    # Intensity
    # -----------------------------

    features = {
        "mean_intensity": np.mean(image),
        "std_intensity": np.std(image),
        "min_intensity": np.min(image),
        "max_intensity": np.max(image),

        "p10": np.percentile(image, 10),
        "p25": np.percentile(image, 25),
        "p50": np.percentile(image, 50),
        "p75": np.percentile(image, 75),
        "p90": np.percentile(image, 90),
    }

    # -----------------------------
    # Entropy
    # -----------------------------

    histogram = cv2.calcHist(
        [image],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    histogram = histogram / (
        histogram.sum() + 1e-8
    )

    features["entropy"] = -np.sum(
        histogram *
        np.log2(histogram + 1e-10)
    )

    # -----------------------------
    # GLCM
    # -----------------------------

    image_32 = (
        image / 8
    ).astype(np.uint8)

    glcm = graycomatrix(
        image_32,
        distances=[1, 2],
        angles=[
            0,
            np.pi / 4,
            np.pi / 2,
            3 * np.pi / 4
        ],
        levels=32,
        symmetric=True,
        normed=True
    )

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation"
    ]:

        values = graycoprops(
            glcm,
            prop
        )

        features[
            f"glcm_{prop}"
        ] = values.mean()

    # -----------------------------
    # LBP
    # -----------------------------

    lbp = local_binary_pattern(
        image,
        P=8,
        R=1,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 11),
        range=(0, 10)
    )

    lbp_hist = lbp_hist.astype(
        np.float32
    )

    lbp_hist /= (
        lbp_hist.sum() + 1e-8
    )

    for i, value in enumerate(lbp_hist):
        features[
            f"lbp_{i}"
        ] = value

    return features


# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

def build_feature_dataframe(
    dataset,
    split_name
):

    rows = []

    for image_path, label in tqdm(
        dataset.samples,
        desc=f"Extracting {split_name}"
    ):

        features = extract_classical_features(
            image_path
        )

        features["label"] = label
        features["image_path"] = image_path

        rows.append(features)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TRAIN_FEATURES_PATH = (
    "./train_classical_features.csv"
)

VAL_FEATURES_PATH = (
    "./val_classical_features.csv"
)


# ------------------------------------------------------------
# Extract
# ------------------------------------------------------------

train_features_df = build_feature_dataframe(
    train_dataset_384,
    "TRAIN"
)

val_features_df = build_feature_dataframe(
    val_dataset_384,
    "VALIDATION"
)


# ------------------------------------------------------------
# Save immediately
# ------------------------------------------------------------

train_features_df.to_csv(
    TRAIN_FEATURES_PATH,
    index=False
)

val_features_df.to_csv(
    VAL_FEATURES_PATH,
    index=False
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("FEATURE TABLES SAVED")
print("=" * 70)

print(
    "Train:",
    train_features_df.shape
)

print(
    "Validation:",
    val_features_df.shape
)

print(
    f"\nSaved train file:\n{TRAIN_FEATURES_PATH}"
)

print(
    f"Saved validation file:\n{VAL_FEATURES_PATH}"
)

print("=" * 70)

In [ ]:
# ============================================================
# CPU-ONLY Classical Feature Extraction
# Rebuild everything needed after Runtime reset
# ============================================================

import os
import numpy as np
import pandas as pd
import cv2

from tqdm.auto import tqdm
from torchvision import datasets

from skimage.feature import (
    graycomatrix,
    graycoprops,
    local_binary_pattern
)

# ------------------------------------------------------------
# 1) Find dataset path
# ------------------------------------------------------------

try:
    DATASET_PATH
except NameError:
    
    DATASET_PATH = get_dataset_path()

print("Dataset path:")
print(DATASET_PATH)

TRAIN_DIR = os.path.join(
    DATASET_PATH,
    "train"
)

VAL_DIR = os.path.join(
    DATASET_PATH,
    "val"
)

# ------------------------------------------------------------
# 2) Create ImageFolder datasets
# ------------------------------------------------------------

train_dataset_cpu = datasets.ImageFolder(
    TRAIN_DIR
)

val_dataset_cpu = datasets.ImageFolder(
    VAL_DIR
)

print("\nDataset check:")
print("Train:", len(train_dataset_cpu))
print("Val  :", len(val_dataset_cpu))

print(
    "Classes:",
    train_dataset_cpu.class_to_idx
)


# ------------------------------------------------------------
# 3) Classical feature extraction
# ------------------------------------------------------------

def extract_classical_features(image_path):

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    # Fixed size for feature extraction
    image = cv2.resize(
        image,
        (224, 224)
    )

    # ----------------------------------------
    # Intensity features
    # ----------------------------------------

    features = {
        "mean_intensity": np.mean(image),
        "std_intensity": np.std(image),
        "min_intensity": np.min(image),
        "max_intensity": np.max(image),

        "p10": np.percentile(image, 10),
        "p25": np.percentile(image, 25),
        "p50": np.percentile(image, 50),
        "p75": np.percentile(image, 75),
        "p90": np.percentile(image, 90),
    }

    # ----------------------------------------
    # Entropy
    # ----------------------------------------

    histogram = cv2.calcHist(
        [image],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    histogram = histogram / (
        histogram.sum() + 1e-8
    )

    features["entropy"] = -np.sum(
        histogram *
        np.log2(histogram + 1e-10)
    )

    # ----------------------------------------
    # GLCM
    # ----------------------------------------

    image_32 = (
        image / 8
    ).astype(np.uint8)

    glcm = graycomatrix(
        image_32,
        distances=[1, 2],
        angles=[
            0,
            np.pi / 4,
            np.pi / 2,
            3 * np.pi / 4
        ],
        levels=32,
        symmetric=True,
        normed=True
    )

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation"
    ]:

        values = graycoprops(
            glcm,
            prop
        )

        features[
            f"glcm_{prop}"
        ] = values.mean()

    # ----------------------------------------
    # LBP
    # ----------------------------------------

    lbp = local_binary_pattern(
        image,
        P=8,
        R=1,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 11),
        range=(0, 10)
    )

    lbp_hist = lbp_hist.astype(
        np.float32
    )

    lbp_hist /= (
        lbp_hist.sum() + 1e-8
    )

    for i, value in enumerate(lbp_hist):
        features[
            f"lbp_{i}"
        ] = value

    return features


# ------------------------------------------------------------
# 4) Build dataframe
# ------------------------------------------------------------

def build_feature_dataframe(
    dataset,
    split_name
):

    rows = []

    for image_path, label in tqdm(
        dataset.samples,
        desc=f"Extracting {split_name}"
    ):

        features = extract_classical_features(
            image_path
        )

        features["label"] = label
        features["image_path"] = image_path

        rows.append(features)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 5) Extract
# ------------------------------------------------------------

train_features_df = build_feature_dataframe(
    train_dataset_cpu,
    "TRAIN"
)

val_features_df = build_feature_dataframe(
    val_dataset_cpu,
    "VALIDATION"
)


# ------------------------------------------------------------
# 6) Save immediately
# ------------------------------------------------------------

TRAIN_FEATURES_PATH = (
    "./train_classical_features.csv"
)

VAL_FEATURES_PATH = (
    "./val_classical_features.csv"
)

train_features_df.to_csv(
    TRAIN_FEATURES_PATH,
    index=False
)

val_features_df.to_csv(
    VAL_FEATURES_PATH,
    index=False
)


# ------------------------------------------------------------
# 7) Final check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE EXTRACTION COMPLETE")
print("=" * 70)

print(
    "Train shape:",
    train_features_df.shape
)

print(
    "Validation shape:",
    val_features_df.shape
)

print(
    "\nTrain class counts:"
)

print(
    train_features_df["label"]
    .value_counts()
    .sort_index()
)

print(
    "\nValidation class counts:"
)

print(
    val_features_df["label"]
    .value_counts()
    .sort_index()
)

print(
    "\nSaved:"
)

print(
    TRAIN_FEATURES_PATH
)

print(
    VAL_FEATURES_PATH
)

print("=" * 70)

In [ ]:
# ============================================================
# Classical Feature Analysis
# Step 3 — Feature Separability
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import kruskal
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Load saved feature tables
# ------------------------------------------------------------

train_features_df = pd.read_csv(
    "./train_classical_features.csv"
)

val_features_df = pd.read_csv(
    "./val_classical_features.csv"
)


# ------------------------------------------------------------
# Select feature columns
# ------------------------------------------------------------

feature_columns = [
    col
    for col in train_features_df.columns
    if col not in ["label", "image_path"]
]

X_train_features = train_features_df[
    feature_columns
]

y_train_features = train_features_df[
    "label"
].values


# ------------------------------------------------------------
# 1. Kruskal-Wallis test
# ------------------------------------------------------------

kruskal_results = []

for feature in feature_columns:

    groups = [
        train_features_df.loc[
            train_features_df["label"] == cls,
            feature
        ].values
        for cls in range(5)
    ]

    statistic, p_value = kruskal(
        *groups
    )

    kruskal_results.append({
        "feature": feature,
        "H_statistic": statistic,
        "p_value": p_value
    })


kruskal_df = pd.DataFrame(
    kruskal_results
).sort_values(
    "p_value"
)


# ------------------------------------------------------------
# 2. Mutual Information
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X_train_features
)

mi_scores = mutual_info_classif(
    X_scaled,
    y_train_features,
    random_state=42
)

mi_df = pd.DataFrame({
    "feature": feature_columns,
    "mutual_information": mi_scores
}).sort_values(
    "mutual_information",
    ascending=False
)


# ------------------------------------------------------------
# 3. Print results
# ------------------------------------------------------------

print("=" * 70)
print("TOP FEATURES — KRUSKAL-WALLIS")
print("=" * 70)

print(
    kruskal_df.head(15).to_string(
        index=False
    )
)


print("\n" + "=" * 70)
print("TOP FEATURES — MUTUAL INFORMATION")
print("=" * 70)

print(
    mi_df.head(15).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 4. Class means for top MI features
# ------------------------------------------------------------

top_features = mi_df.head(
    8
)["feature"].tolist()

class_means = (
    train_features_df
    .groupby("label")[top_features]
    .mean()
)

class_means.index = [
    "KL0_Normal",
    "KL1_Doubtful",
    "KL2_Mild",
    "KL3_Moderate",
    "KL4_Severe"
]

print("\n" + "=" * 70)
print("CLASS MEANS — TOP FEATURES")
print("=" * 70)

print(
    class_means.to_string()
)

print("=" * 70)

In [ ]:
# ============================================================
# Classical ML Baseline
# Random Forest on GLCM + LBP + Intensity Features
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score
)


# ------------------------------------------------------------
# Load feature tables
# ------------------------------------------------------------

train_features_df = pd.read_csv(
    "./train_classical_features.csv"
)

val_features_df = pd.read_csv(
    "./val_classical_features.csv"
)


# ------------------------------------------------------------
# Feature columns
# ------------------------------------------------------------

feature_columns = [
    col
    for col in train_features_df.columns
    if col not in ["label", "image_path"]
]


X_train = train_features_df[
    feature_columns
]

y_train = train_features_df[
    "label"
]

X_val = val_features_df[
    feature_columns
]

y_val = val_features_df[
    "label"
]


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("=" * 70)
print("TRAINING RANDOM FOREST")
print("=" * 70)

rf_model.fit(
    X_train,
    y_train
)

print("Training complete.")


# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

rf_predictions = rf_model.predict(
    X_val
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_val,
    rf_predictions
)

balanced_accuracy = balanced_accuracy_score(
    y_val,
    rf_predictions
)

macro_f1 = f1_score(
    y_val,
    rf_predictions,
    average="macro",
    zero_division=0
)

mae = mean_absolute_error(
    y_val,
    rf_predictions
)

qwk = cohen_kappa_score(
    y_val,
    rf_predictions,
    weights="quadratic"
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

print("\n" + "=" * 70)
print("RANDOM FOREST — VALIDATION RESULTS")
print("=" * 70)

print(
    f"Accuracy           : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy  : {balanced_accuracy:.4f}"
)

print(
    f"Macro F1           : {macro_f1:.4f}"
)

print(
    f"MAE                : {mae:.4f}"
)

print(
    f"QWK                : {qwk:.4f}"
)


print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_val,
        rf_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

cm = confusion_matrix(
    y_val,
    rf_predictions
)

print(
    "Rows = True class"
)

print(
    "Cols = Predicted class\n"
)

print(
    "             " +
    "  ".join(
        f"{name[:9]:>9}"
        for name in class_names
    )
)

for i, row in enumerate(cm):

    print(
        f"{class_names[i]:<10}" +
        "  ".join(
            f"{value:9d}"
            for value in row
        )
    )


# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\n" + "=" * 70)
print("TOP 15 RANDOM FOREST FEATURES")
print("=" * 70)

print(
    importance_df.head(15).to_string(
        index=False
    )
)

print("=" * 70)

In [ ]:
# ============================================================
# ROI Classical Features
# Step 1 — Extract GLCM + LBP + Intensity from Center ROI
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from torchvision import datasets

# ------------------------------------------------------------
# Rebuild datasets if needed
# ------------------------------------------------------------


DATASET_PATH = get_dataset_path()

TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR   = os.path.join(DATASET_PATH, "val")

train_dataset_roi = datasets.ImageFolder(TRAIN_DIR)
val_dataset_roi   = datasets.ImageFolder(VAL_DIR)


# ------------------------------------------------------------
# Feature extraction from center ROI
# ------------------------------------------------------------

def extract_roi_features(image_path, crop_ratio=0.80):

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    # Resize to common size
    image = cv2.resize(
        image,
        (224, 224)
    )

    # --------------------------------------------------------
    # Center crop
    # --------------------------------------------------------

    crop_size = int(
        224 * crop_ratio
    )

    start = (
        224 - crop_size
    ) // 2

    end = start + crop_size

    roi = image[
        start:end,
        start:end
    ]

    # --------------------------------------------------------
    # Intensity features
    # --------------------------------------------------------

    features = {

        "roi_mean_intensity":
            np.mean(roi),

        "roi_std_intensity":
            np.std(roi),

        "roi_min_intensity":
            np.min(roi),

        "roi_max_intensity":
            np.max(roi),

        "roi_p10":
            np.percentile(roi, 10),

        "roi_p25":
            np.percentile(roi, 25),

        "roi_p50":
            np.percentile(roi, 50),

        "roi_p75":
            np.percentile(roi, 75),

        "roi_p90":
            np.percentile(roi, 90)
    }


    # --------------------------------------------------------
    # Entropy
    # --------------------------------------------------------

    histogram = cv2.calcHist(
        [roi],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    histogram = histogram / (
        histogram.sum() + 1e-8
    )

    features["roi_entropy"] = -np.sum(
        histogram *
        np.log2(histogram + 1e-10)
    )


    # --------------------------------------------------------
    # GLCM
    # --------------------------------------------------------

    roi_32 = (
        roi / 8
    ).astype(np.uint8)

    glcm = graycomatrix(
        roi_32,
        distances=[1, 2],
        angles=[
            0,
            np.pi / 4,
            np.pi / 2,
            3 * np.pi / 4
        ],
        levels=32,
        symmetric=True,
        normed=True
    )

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation"
    ]:

        values = graycoprops(
            glcm,
            prop
        )

        features[
            f"roi_glcm_{prop}"
        ] = values.mean()


    # --------------------------------------------------------
    # LBP
    # --------------------------------------------------------

    lbp = local_binary_pattern(
        roi,
        P=8,
        R=1,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 11),
        range=(0, 10)
    )

    lbp_hist = lbp_hist.astype(
        np.float32
    )

    lbp_hist /= (
        lbp_hist.sum() + 1e-8
    )

    for i, value in enumerate(lbp_hist):

        features[
            f"roi_lbp_{i}"
        ] = value

    return features


# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

def build_roi_feature_dataframe(
    dataset,
    split_name
):

    rows = []

    for image_path, label in tqdm(
        dataset.samples,
        desc=f"Extracting ROI {split_name}"
    ):

        features = extract_roi_features(
            image_path,
            crop_ratio=0.80
        )

        features["label"] = label
        features["image_path"] = image_path

        rows.append(features)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Extract
# ------------------------------------------------------------

train_roi_df = build_roi_feature_dataframe(
    train_dataset_roi,
    "TRAIN"
)

val_roi_df = build_roi_feature_dataframe(
    val_dataset_roi,
    "VALIDATION"
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

TRAIN_ROI_PATH = (
    "./train_roi_classical_features.csv"
)

VAL_ROI_PATH = (
    "./val_roi_classical_features.csv"
)

train_roi_df.to_csv(
    TRAIN_ROI_PATH,
    index=False
)

val_roi_df.to_csv(
    VAL_ROI_PATH,
    index=False
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

feature_cols_roi = [
    col
    for col in train_roi_df.columns
    if col not in [
        "label",
        "image_path"
    ]
]

print("=" * 70)
print("ROI FEATURE EXTRACTION COMPLETE")
print("=" * 70)

print(
    "Train shape:",
    train_roi_df.shape
)

print(
    "Validation shape:",
    val_roi_df.shape
)

print(
    "Number of ROI features:",
    len(feature_cols_roi)
)

print(
    "\nSaved:"
)

print(TRAIN_ROI_PATH)
print(VAL_ROI_PATH)

print("=" * 70)

In [ ]:
# ============================================================
# ROI Classical ML Comparison
# KNN vs SVM vs Random Forest
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score,
    classification_report
)


# ------------------------------------------------------------
# 1. Load ROI feature tables
# ------------------------------------------------------------

train_roi_df = pd.read_csv(
    "./train_roi_classical_features.csv"
)

val_roi_df = pd.read_csv(
    "./val_roi_classical_features.csv"
)


# ------------------------------------------------------------
# 2. Select features
# ------------------------------------------------------------

feature_cols = [
    col
    for col in train_roi_df.columns
    if col not in [
        "label",
        "image_path"
    ]
]

X_train = train_roi_df[
    feature_cols
]

y_train = train_roi_df[
    "label"
].values

X_val = val_roi_df[
    feature_cols
]

y_val = val_roi_df[
    "label"
].values


print("=" * 70)
print("ROI CLASSICAL ML COMPARISON")
print("=" * 70)

print(
    f"Number of features: {len(feature_cols)}"
)

print(
    f"Train samples: {len(X_train)}"
)

print(
    f"Validation samples: {len(X_val)}"
)


# ------------------------------------------------------------
# 3. Define models
# ------------------------------------------------------------

models_dict = {

    "KNN": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            KNeighborsClassifier(
                n_neighbors=15,
                weights="distance",
                metric="euclidean"
            )
        )
    ]),

    "SVM_RBF": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            SVC(
                kernel="rbf",
                C=2.0,
                gamma="scale",
                class_weight="balanced"
            )
        )
    ]),

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}


# ------------------------------------------------------------
# 4. Train and evaluate
# ------------------------------------------------------------

results = []

predictions_dict = {}

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

for model_name, model in models_dict.items():

    print("\n" + "=" * 70)
    print(f"TRAINING: {model_name}")
    print("=" * 70)

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_val
    )

    predictions_dict[
        model_name
    ] = predictions

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    balanced_acc = balanced_accuracy_score(
        y_val,
        predictions
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    qwk = cohen_kappa_score(
        y_val,
        predictions,
        weights="quadratic"
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced_acc,
        "Macro_F1": macro_f1,
        "MAE": mae,
        "QWK": qwk
    })

    print(
        f"Accuracy          : {accuracy:.4f}"
    )

    print(
        f"Balanced Accuracy : {balanced_acc:.4f}"
    )

    print(
        f"Macro F1          : {macro_f1:.4f}"
    )

    print(
        f"MAE               : {mae:.4f}"
    )

    print(
        f"QWK               : {qwk:.4f}"
    )


# ------------------------------------------------------------
# 5. Comparison table
# ------------------------------------------------------------

results_df = pd.DataFrame(
    results
).sort_values(
    "Macro_F1",
    ascending=False
)

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 6. Best classical model
# ------------------------------------------------------------

best_model_name = results_df.iloc[0]["Model"]

best_predictions = predictions_dict[
    best_model_name
]

print("\n" + "=" * 70)
print(
    f"BEST CLASSICAL MODEL: {best_model_name}"
)
print("=" * 70)

print(
    classification_report(
        y_val,
        best_predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


print("Confusion Matrix:")

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_val,
    best_predictions
)

print(cm)

print("=" * 70)

In [ ]:
# ============================================================
# Hybrid Experiment
# Step 1 — Extract 512-D Deep Features from Best CNN
# CPU version
# ============================================================

import os
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn

from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1. Dataset path
# ------------------------------------------------------------


DATASET_PATH = get_dataset_path()

TRAIN_DIR = os.path.join(
    DATASET_PATH,
    "train"
)

VAL_DIR = os.path.join(
    DATASET_PATH,
    "val"
)


# ------------------------------------------------------------
# 2. CLAHE transform
# ------------------------------------------------------------

class CLAHETransform:

    def __init__(
        self,
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):

        img = np.array(img)

        if len(img.shape) == 3:
            img = cv2.cvtColor(
                img,
                cv2.COLOR_RGB2GRAY
            )

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        img = clahe.apply(img)

        return Image.fromarray(img)


# ------------------------------------------------------------
# 3. Same evaluation preprocessing as best CNN
# ------------------------------------------------------------

deep_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 4. Dataset based on saved feature-table image paths
# ------------------------------------------------------------

class DeepFeatureDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image_path = self.df.loc[
            idx,
            "image_path"
        ]

        label = int(
            self.df.loc[
                idx,
                "label"
            ]
        )

        image = Image.open(
            image_path
        ).convert("L")

        image = self.transform(
            image
        )

        return image, label


# ------------------------------------------------------------
# 5. Load saved feature tables
# ------------------------------------------------------------

train_roi_df = pd.read_csv(
    "./train_roi_classical_features.csv"
)

val_roi_df = pd.read_csv(
    "./val_roi_classical_features.csv"
)


# ------------------------------------------------------------
# 6. Build datasets
# ------------------------------------------------------------

train_deep_dataset = DeepFeatureDataset(
    train_roi_df,
    deep_transform
)

val_deep_dataset = DeepFeatureDataset(
    val_roi_df,
    deep_transform
)


train_deep_loader = DataLoader(
    train_deep_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)

val_deep_loader = DataLoader(
    val_deep_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)


# ------------------------------------------------------------
# 7. Rebuild ResNet18
# ------------------------------------------------------------

device_cpu = torch.device("cpu")

model_deep = models.resnet18(
    weights=None
)

model_deep.fc = nn.Linear(
    model_deep.fc.in_features,
    5
)


# ------------------------------------------------------------
# 8. Load BEST Sampler CNN checkpoint
# ------------------------------------------------------------

BEST_CNN_PATH = (
    "./best_resnet18_oa_sampler.pth"
)

checkpoint_deep = torch.load(
    BEST_CNN_PATH,
    map_location=device_cpu,
    weights_only=False
)

model_deep.load_state_dict(
    checkpoint_deep["model_state_dict"]
)

model_deep = model_deep.to(
    device_cpu
)

model_deep.eval()


# ------------------------------------------------------------
# 9. Remove final classifier
#    ResNet18 -> 512-D feature vector
# ------------------------------------------------------------

model_deep.fc = nn.Identity()

model_deep.eval()


# ------------------------------------------------------------
# 10. Feature extraction function
# ------------------------------------------------------------

def extract_deep_features(
    loader,
    split_name
):

    features = []
    labels = []

    with torch.no_grad():

        for images, batch_labels in tqdm(
            loader,
            desc=f"Deep features — {split_name}"
        ):

            images = images.to(
                device_cpu
            )

            outputs = model_deep(
                images
            )

            features.append(
                outputs.numpy()
            )

            labels.append(
                batch_labels.numpy()
            )

    features = np.concatenate(
        features,
        axis=0
    )

    labels = np.concatenate(
        labels,
        axis=0
    )

    return features, labels


# ------------------------------------------------------------
# 11. Extract TRAIN features
# ------------------------------------------------------------

train_deep_features, train_deep_labels = (
    extract_deep_features(
        train_deep_loader,
        "TRAIN"
    )
)


# ------------------------------------------------------------
# 12. Extract VALIDATION features
# ------------------------------------------------------------

val_deep_features, val_deep_labels = (
    extract_deep_features(
        val_deep_loader,
        "VALIDATION"
    )
)


# ------------------------------------------------------------
# 13. Save to disk
# ------------------------------------------------------------

np.savez_compressed(
    "./train_deep_features.npz",
    features=train_deep_features,
    labels=train_deep_labels
)

np.savez_compressed(
    "./val_deep_features.npz",
    features=val_deep_features,
    labels=val_deep_labels
)


# ------------------------------------------------------------
# 14. Verify
# ------------------------------------------------------------

print("=" * 70)
print("DEEP FEATURE EXTRACTION COMPLETE")
print("=" * 70)

print(
    "Train deep feature shape:",
    train_deep_features.shape
)

print(
    "Validation deep feature shape:",
    val_deep_features.shape
)

print(
    "Train labels:",
    train_deep_labels.shape
)

print(
    "Validation labels:",
    val_deep_labels.shape
)

print(
    "\nSaved:"
)

print(
    "./train_deep_features.npz"
)

print(
    "./val_deep_features.npz"
)

print("=" * 70)

In [ ]:
# ============================================================
# Check whether any saved model checkpoints still exist
# ============================================================

import os

search_roots = [
    ".",
    "./drive/MyDrive"
]

extensions = (
    ".pth",
    ".pt",
    ".ckpt"
)

found = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):

        for filename in filenames:

            if filename.endswith(extensions):

                full_path = os.path.join(
                    dirpath,
                    filename
                )

                found.append(full_path)


print("=" * 70)
print("CHECKPOINT SEARCH")
print("=" * 70)

if found:

    for path in found:
        print(path)

else:

    print(
        "No PyTorch checkpoint files were found."
    )

print("=" * 70)